# gpu/cpu check

In [ ]:
import tensorflow as tf
from tensorflow.python.client import device_lib
tf.test.gpu_device_name()
device_lib.list_local_devices()

In [ ]:
"""the same opener tasks 1-3 use, so this notebook drops straight into that Colab
setup. Guarded, because it also has to run on Kaggle, where /kaggle/input is
already mounted read-only and google.colab does not exist."""
import sys
if 'google.colab' in sys.modules:
    from google.colab import drive
    drive.mount('/content/drive')
else:
    print('not on Colab - skipping drive.mount. On Kaggle the attached datasets')
    print('are already mounted under /kaggle/input; the next cell finds them.')

---
### ⚠️ **Status of this notebook**

**This is a complete implementation that has NOT been run end to end yet.**
Every cell below is written and ready to execute, but all outputs are cleared and
**no result numbers are filled in anywhere** - no accuracy, no Jaccard, no coefficients, no
rationale text, no intersection size, no confusion matrix. The cells show the code that *would*
produce those numbers. Do not read any figure in this notebook as a measured result, because
there are none.

The only measured numbers stated anywhere in this notebook are:

* **0.615** - test accuracy of the black-box baseline, from `task3-lesion-classification.ipynb`'s
  own saved output (small `Sequential` CNN, 32x32 input, 7 classes).
* **0.793 / 0.873** - Jaccard / Dice of the ResNet50-U-Net in `task1-lesion-segmentation.ipynb`.
* **0.292** - the ISIC 2018 Task 2 *winner's* average Jaccard, published, not mine
  (<https://arxiv.org/abs/2104.01641>), against a Task 1 leaderboard topping out around **0.80**
  (<https://challenge.isic-archive.com/leaderboards/2018/>).
* **2594 / 10015 / 0**, and the id ranges - the dataset counts in the data-join section. These
  are properties of the published ISIC 2018 distribution, read off the official archives and the
  ISIC Archive API, not results of this notebook. It recomputes all of them at runtime and prints
  its own answer, which is the number to trust. (**3694 / 18470** are what a probe of one *Kaggle
  mirror* found; the discrepancy is discussed where it appears.)
* published clinical figures (odds ratios, sensitivities, ICCs), every one attributed inline.

Everything else is empty on purpose.

Run order: `env` -> `imports` -> `paths / attributes` -> `clinical wording` -> `io helpers` ->
`the data join` -> `route` -> `concept vector` -> `load g + thresholds` -> `concept matrix` ->
`f()` -> `regimes` -> `fidelity` -> `rationale` -> `sensitivity` -> `comparison`.

---


# **Concept bottleneck: chaining Task 2 into Task 3**

The three notebooks in this repo do not talk to each other. Task 1 segments a lesion, Task 2
segments five dermoscopic attributes inside it, Task 3 classifies a diagnosis - three separate
models, three separate `.h5` files, nothing consumed downstream. The repo README lists that as
regret #5: "Segmentation and classification never met."

This notebook makes them meet, and specifically makes them meet in the one arrangement where
chaining them buys something other than a marginal accuracy point:

```
             image
               |
               v
   [ Task-2 multi-label U-Net ]        <- g(.), frozen. 5 attribute masks, (256,256,5)
               |
               v
     interpretable concept vector      <- 20 named scalars, every one printable
    (per attribute: present, area
     fraction, focus count, asymmetry)
               |
               v
   [ multinomial logistic regression ] <- f(.), ~140 coefficients, no hidden layer
               |
               v
    diagnosis  +  "MEL, because ..."   <- the coefficients ARE the explanation
```

**The point is not accuracy.** A 32x32 CNN already gets 0.615 on this task and a properly
trained transfer-learning model would beat both it and this. The point is that when this model
is wrong you can ask it *why* and get an answer in the vocabulary a dermatologist already uses -
"melanoma, because streaks are present, asymmetrically distributed, over 4% of the frame, and a
negative network is present" - and then check each of those clauses independently against the
image. A saliency heatmap over a black box cannot be checked; it can only be looked at.

There is a second, sharper reason to want this. Because every bit of information reaching the
diagnosis passes through a named, thresholded quantity, the pipeline supports questions that are
simply not askable of a black box:

* **Was this error a concept error or a reasoning error?** Did `g` fail to see the streaks, or
  did `f` draw the wrong conclusion from correctly-seen streaks?
* **Is the model's reasoning clinically defensible at all**, independent of whether it happens
  to be right on this dataset?
* **What happens if I correct one concept?** Overwrite it and re-predict.

The last one turns out to be exactly what this dataset will not let us do, for a reason that is
worth the whole section below.


# **What a concept bottleneck model is**

Koh et al., *Concept Bottleneck Models*, ICML 2020 (PMLR 119:5338-5348) -
<https://arxiv.org/abs/2007.04612>.

The construction is small. Instead of learning `x -> y` directly, learn two functions

```
g : x -> c        (input to concepts,    here: image -> 5 dermoscopic attributes)
f : c -> y        (concepts to label,    here: concepts -> 1 of 7 diagnoses)
```

and predict `y_hat = f(g(x))`. In the paper's words the prediction "relies on the input `x`
entirely through the bottleneck `c_hat = g(x)`, which we train to align component-wise to the
concepts `c`". *Component-wise* is the load-bearing part: dimension `j` of the bottleneck is
supervised to mean one specific named thing, which is what makes reading it off legitimate.

The three training regimes, with the paper's own equations:

| regime | how it is fitted | what it buys | what it costs |
|---|---|---|---|
| **Independent** | `f = argmin sum L_Y(f(c); y)` on the **true** concepts, and `g = argmin sum L_C(g(x); c)`, with neither knowing the other exists | `f` sees clean concepts, so its coefficients are a statement about the *concepts themselves*, not about one particular `g`'s error pattern. Best behaved under test-time intervention, because corrected concepts are exactly the distribution it was trained on | train/test mismatch: "while `f` is trained using the true `c`, at test time it still takes `g(x)` as input". Needs `c` and `y` on the **same** images |
| **Sequential** | fit `g` first, freeze it, then `f = argmin sum L_Y(f(g(x)); y)` on the **predicted** concepts | `f` can adapt to how badly `g` does on each concept - it can learn to discount a concept `g` is bad at. Needs only labels, never `c`, once `g` exists | `f`'s coefficients are now partly a statement about `g`'s biases. Interventions push the input off the distribution `f` was fitted on, so correcting concepts can *hurt* |
| **Joint** | one objective: `f, g = argmin sum_i [ L_Y(f(g(x_i)); y_i) + sum_j lambda * L_C_j(g(x_i); c_j_i) ]` | highest accuracy of the three in the paper's CUB experiment, because the concepts get "refined to improve predictive performance" | that refinement is the problem - `g`'s outputs drift away from meaning what they are named. Needs `c` and `y` on the same images |

`lambda` is what Koh et al. call the **task-concept tradeoff hyperparameter**, and the two
limits are worth memorising because they are easy to get backwards:

* `lambda -> 0` is **the standard black-box model** - a network with a narrow layer in it and
  no reason for that layer to mean anything.
* `lambda -> infinity` is **the sequential model** - concepts fitted with no regard for the
  downstream task at all.

Independent is *not* a limit of the joint objective; it is a different procedure.
They used `lambda = 1` for OAI and `lambda = 0.01` for CUB, chosen as "the model that has the
highest task accuracy while maintaining high concept accuracy on the validation set".

**Does the bottleneck cost accuracy?** In their experiments, not much. On OAI the joint and
sequential CBMs (0.418 RMSE) actually *beat* the standard model (0.441). On CUB the CBMs are
worse - best CBM 0.199 error against 0.175 for the standard model - and they say so: the
sequential and independent bottlenecks "are worse in 0-1 error than the standard model, though
the joint model ... closes most of the gap". So "CBMs match black boxes" is true on one of
their two datasets and approximately true on the other. It is not a law, and this notebook
should not be read as expecting it.

**Which regime fits *this* repo?** Sequential, and not as a compromise. Task 2 already
produces a trained U-Net `g` that maps an image to five attribute masks; sequential is the
regime that consumes exactly that - a frozen, already-fitted `g` - and needs nothing from the
attribute ground truth afterwards. Independent and joint both require `c` and `y` on the same
image, which is precisely the thing the data-join section below has to go and measure.


# **The known failure mode: concept leakage**

A concept bottleneck is only as honest as the claim that its dimensions mean what they are
labelled. That claim fails in a specific, well-documented way.

**Leakage.** Mahinpei, Clark, Lage, Doshi-Velez, Pan, *Promises and Pitfalls of Black-Box
Concept Learning Models*, 2021 - <https://arxiv.org/abs/2106.13314>. Concept representations
"encode information beyond the pre-defined concepts", which "renders the interpretation of the
downstream prediction misleading". The mechanism is not a bug, it is arithmetic: a soft concept
score is a distance from a decision surface, so it "necessarily encode[s] the distribution of
the data along axes perpendicular to decision surfaces", and the downstream classifier happily
uses that extra information.

Their cleanest demonstration: predict the *parity* of an MNIST digit through a two-concept
bottleneck, "is a 4" and "is a 5", on a subset containing **no 4s and no 5s at all**. The
concepts are provably irrelevant, so the ceiling should be 50%. The downstream model reaches
**69%**. The "is a 4" activation correlates -0.72 with the first principal component of the
data. The bottleneck was smuggling the image through.

Two things about this that matter for the design below:

1. **Leakage is not caused by joint training.** That was the earlier hypothesis (Margeloiu et
   al., *Do Concept Bottleneck Models Learn as Intended?*, ICLR 2021 workshop -
   <https://arxiv.org/abs/2105.04289>, which found a joint CBM with a **single** concept in the
   bottleneck still beat a one-concept oracle, i.e. "there is no concept bottleneck"). Mahinpei
   et al. show leakage "occurs even without joint training" - sequential training leaks too.
   Joint training makes it worse, it does not create it.
2. **The interpretive consequence is stated plainly by the authors** and is worth quoting
   because it is exactly the mistake this notebook could invite: "the fact that the predicted
   probability of an increase in tremors is highly indicative of Parkinson's disease does not
   imply that there is significant correlation between an increase in tremors and Parkinson's
   in the data."

So: a large positive coefficient on `streaks__area_frac` for MEL below is **not** evidence that
streaks indicate melanoma. It is evidence that this `g`'s streak channel carries information
this `f` finds useful for MEL, which may be streaks or may be anything correlated with them
that survived the U-Net.

**Hard concepts leak less.** Havasi, Parbhoo, Doshi-Velez, *Addressing Leakage in Concept
Bottleneck Models*, NeurIPS 2022 -
<https://papers.neurips.cc/paper_files/paper/2022/file/944ecf65a46feb578a43abfd5cddd960-Paper-Conference.pdf>
- state the failure most crisply: with soft concepts the label predictor "only needs to encode
the class label in the soft concept probabilities", and so "a human supervisor can never be
sure if the model is predicting a certain concept because it is likely to be present, or
because it is encoding for something else." Their CUB numbers are the sharpest exhibit of the
tradeoff in the literature: a soft joint CBM at `lambda = 0.01` gets **82.1%** label accuracy
with **0.1%** concept accuracy (their concept accuracy = all concepts simultaneously correct).
The hard-concept variant gets 79.5% / 68.0%.

That is a direct argument for the `present` dimensions in the concept vector below: a
thresholded, binary "is this attribute here" leaks less than a continuous score. It is also an
argument for *keeping* the continuous ones for a separate reason - clinical criteria are about
extent and irregularity, not only presence - so the vector below deliberately carries both, and
the honest reading is that the four dimensions per attribute differ in how trustworthy they are
as explanations, with `present` the most trustworthy and `area_frac` the least.

# **Side channels, and the tradeoff they reintroduce**

The standard fix when a bottleneck costs too much accuracy is to route around it.

* **Hybrid CBM** - add `gamma` *unsupervised* dimensions alongside the `k` supervised ones and
  concatenate. (The construction is evaluated in Mahinpei et al.; the name comes from the CEM
  paper below.)
* **Side-channel CBM** - Havasi et al. 2022, above: model `L` explicitly unknown latent
  concepts `z` alongside the known `c`, so the model "can capture information about `y` present
  in `x` but not present in `c`". They add the nice touch of an amortization network that lets
  `z` be *marginalised out* at prediction time, so you get a switch rather than a permanent
  compromise.
* **PCBM-h** - Yuksekgonul, Wang, Zou, *Post-hoc Concept Bottleneck Models*, ICLR 2023 -
  <https://arxiv.org/abs/2205.15480>. Fit the interpretable predictor on the concept
  projection, freeze it, then fit a linear **residual** `r` on the *full backbone embedding* to
  soak up what the bottleneck missed: `min_r L( g(f_C(x)) + r(f(x)), y )`.
* **Concept Embedding Models** - Espinosa Zarlenga et al., NeurIPS 2022 -
  <https://arxiv.org/abs/2209.09056>. Give each concept an `m`-dimensional *supervised*
  embedding instead of a scalar, so extra capacity is added without a concept-agnostic channel.

The tradeoff each of these reintroduces is the same one, and every one of these papers names it
about its own method:

> **Once information can reach the label without passing through a named concept, intervening
> on the concepts stops being a test of what the model believes.**

Concretely. CEM on Hybrid CBMs: extra capacity "comes at the cost of their interpretability and
their responsiveness to concept interventions", and "interventions in Hybrid CBM bottlenecks
have little effect on their predictive accuracy". PCBM on itself: "PCBM-h is a 'less'
interpretable but more powerful variant of PCBM" - and when they try to *edit* a model to
remove a spurious concept, "in PCBM, we can remove the concept from the model, but in PCBM-h,
there may still be leftover information about the spurious concept in the residual part of the
model." Havasi et al. give the user the choice explicitly: use the side channel "for
applications where accuracy is high-priority and partial explanations are sufficient",
marginalise it out "for applications where the human operator must know the full set of
concepts contributing to the prediction."

The accuracy at stake is real. PCBM's table, on skin data specifically: original model 0.963
AUROC on HAM10000, PCBM 0.947, PCBM-h 0.962; on their ISIC task 0.821 / 0.736 / 0.801. So the
side channel bought back roughly 6.5 AUROC points on ISIC - and bought back with it the
inability to say that removing a concept removes its influence.

**This notebook deliberately has no side channel.** Not because a side channel is wrong, but
because the whole point of the exercise is the audit: the intervention experiment and the
concept/reasoning failure attribution near the end are only meaningful while every bit of
information reaching the diagnosis is a named, thresholded, clinically-labelled quantity. The
place a side channel would go is marked in the concept-vector cell - it would be one extra
block of features from the raw image - and it is left out on purpose. If a future version adds
one, every intervention result below becomes uninterpretable and must be deleted rather than
re-run.

**One more caution, from PCBM's own results:** on HAM10000 they needed only 8 concepts to lose
almost nothing (0.963 -> 0.947). This notebook has 5 attributes producing 20 numbers. There is
no reason to expect the 5 ISIC attributes to be a *complete* description of what separates 7
diagnoses - AKIEC, BCC, DF and VASC are not melanocytic lesions and are largely not described
by these five melanocytic-pattern attributes at all. An incomplete concept set is precisely the
condition under which Havasi et al. predict leakage and under which Koh et al. observed the CUB
accuracy gap. Expect the bottleneck to cost accuracy here, and expect it to cost most on the
non-melanocytic classes.


# **Are these five things actually clinical criteria?**

The whole claim of this notebook is that the bottleneck speaks a dermatologist's language. That
claim has to be checked against the dermatology literature rather than assumed, and checking it
turns up one problem serious enough that it shapes the concept-vector design two sections down.

## The five, one at a time

| ISIC attribute | recognised criterion? | in the 7-point checklist? | direction | strongest number I could source |
|---|---|---|---|---|
| **Pigment network** | Yes - central to pattern analysis. Brown lines in a grid; the lines are melanin along elongated rete ridges | Yes, criterion #1, **atypical pigment network**, a *major* criterion (2 points) | **Depends completely on typical vs atypical.** *Typical* (uniform line width and colour) is "common in benign melanocytic nevi and non-melanocytic lesions (dermatofibromas, ink spot lentigo)". *Atypical* (lines varying in size, colour, thickness or distribution) points to dysplastic nevi and superficial spreading melanoma | atypical network **OR 2.8 (95% CI 2.4-3.4)** for melanoma |
| **Negative network** | Yes, but weak - lighter serpiginous grid lines between elongated hyperpigmented globules | **No.** Not one of the seven | Melanoma-leaning but *specific and insensitive*, and it is common in benign lesions too: present in 34.6% of melanomas, **28.8% of Spitz nevi** (OR 1.1, not significant) and 18.2% of ordinary nevi | **sensitivity 34.6%, specificity ~77%, OR 1.8 (1.3-2.7)** (Pizzichetta 2013); OR 1.4 (1.1-1.8) (Carrera 2016) |
| **Streaks** | Yes - linear pigmented projections at the lesion periphery (radial streaming and pseudopods) | Yes, criterion #4, **irregular streaks**, a *minor* criterion (1 point) | **Distribution decides, not presence.** Streaks distributed *symmetrically* around the whole lesion favour a **Reed / Spitz (benign) nevus**; *asymmetric* distribution is what means melanoma should be excluded | streaks **OR 1.5 (1.3-1.8)**; pseudopods **OR 2.1 (1.7-2.5)** |
| **Milia-like cysts** | Yes, but as a *keratinocytic* clue, not a melanoma criterion. Round whitish/yellowish structures corresponding to intraepidermal keratin pseudocysts | **No.** Not in the 7-point checklist, not in the ABCD structures, not in Menzies | Benign-leaning **towards seborrheic keratosis** - but only for one subtype. "**Cloudy**" cysts (larger, hazier) reached 99.1% specificity for seborrheic keratosis in an SK-vs-melanoma comparison; "**starry**" cysts (small, bright, sharp) occurred in *both* SK and melanoma and had limited value. They also occur in congenital-pattern nevi | cloudy MLC **99.1% specificity** for SK within an SK-vs-melanoma set of 221 melanomas / 175 SKs |
| **Globules** (incl. dots) | Yes - nests of pigmented melanocytes; >0.1 mm, round-to-oval, well demarcated | Yes, criterion #5, **irregular dots/globules**, a *minor* criterion (1 point) | **Regular vs irregular reverses the reading.** Uniform in size, shape and colour, evenly distributed or symmetric around the whole perimeter = benign pattern. Varying in size/shape/colour, unevenly distributed, *focally* peripheral = melanoma pattern | irregular black dots **OR 1.8 (1.5-2.1)**; irregular black globules **OR 1.9 (1.5-2.3)** |

Sources for the table: dermoscopedia's structure pages -
[pigment network](https://dermoscopedia.org/Pigment_network),
[negative network](https://dermoscopedia.org/Negative_network),
[streaks](https://dermoscopedia.org/Streaks),
[milia-like cysts](https://dermoscopedia.org/Milia_like_cysts),
[dermoscopic structures](https://dermoscopedia.org/Dermoscopic_structures);
odds ratios and reliability from Carrera et al., *Validity and Reliability of Dermoscopic
Criteria Used to Differentiate Nevi From Melanoma*, JAMA Dermatol 2016;152(7):798-806 -
<https://pmc.ncbi.nlm.nih.gov/articles/PMC5451089/>;
negative-network figures from Pizzichetta et al., *Negative pigment network: an additional
dermoscopic feature for the diagnosis of melanoma*, J Am Acad Dermatol 2013;68(4):552-559 -
<https://pubmed.ncbi.nlm.nih.gov/23062610/>;
cloudy/starry cysts from J Eur Acad Dermatol Venereol 2011 -
<https://pubmed.ncbi.nlm.nih.gov/21923811/>.

Two citation traps worth recording, since both are widespread:

* The often-quoted "negative network: 22% sensitivity, 95% specificity" is **not** Pizzichetta.
  Pizzichetta's own numbers are 34.6% / ~77%; they *cite* 22%/95% from Menzies, Ingvar,
  McCarthy, Melanoma Res 1996;6:55-62. Secondary sources routinely merge the two.
* Pizzichetta's text says specificity 77.4% while its own Table II says 77.2%. Write "~77%".

## The 7-point checklist

Argenziano, Fabbrocini, Carli, De Giorgi, Sammarco, Delfino, *Epiluminescence microscopy for
the diagnosis of doubtful melanocytic skin lesions. Comparison of the ABCD rule of dermatoscopy
and a new 7-point checklist based on pattern analysis*, **Arch Dermatol. 1998;134(12):1563-1570**
- see <https://dermoscopedia.org/Seven_Point_Checklist>.

| # | criterion | points |
|---|---|---|
| 1 | **Atypical pigment network** | 2 (major) |
| 2 | Blue-white veil | 2 (major) |
| 3 | Atypical vascular pattern | 2 (major) |
| 4 | **Irregular streaks** | 1 (minor) |
| 5 | **Irregular dots / globules** | 1 (minor) |
| 6 | Irregular blotches | 1 (minor) |
| 7 | Regression structures | 1 (minor) |

Score **>= 3** means melanoma / excise. Original validation on 342 melanocytic lesions (117
melanomas, 225 atypical nevi): **sensitivity 95%, specificity 75%**. A revised version
(Argenziano et al., Br J Dermatol 2011;164(4):785-790 -
<https://onlinelibrary.wiley.com/doi/abs/10.1111/j.1365-2133.2010.10194.x>) drops the
major/minor weighting to one point each and lowers the threshold to >= 1 feature, giving
sensitivity 85-93% / specificity 45-48%.

**Do not use the 95%/75% figures as an expectation.** Prospective and independent evaluations
are much less flattering: a 10-year prospective surveillance study found **sensitivity 62%,
specificity 97%** at threshold >= 3 (<https://pubmed.ncbi.nlm.nih.gov/20226567/>), and
Carrera's web-based reader study found **sensitivity 70.6%, specificity 57.5%**. This matters
here because it puts a ceiling on how good a *correct* checklist-style reasoner can be, before
any question of whether this notebook's `g` finds the structures at all.

## The problem that shapes the design: ISIC's labels are unqualified

Read the table again and notice what every single "depends" has in common.

**Three of the five ISIC attributes map onto a 7-point criterion, and all three map only
conditionally** - onto *atypical* pigment network, *irregular* streaks, *irregular*
dots/globules. The remaining two, negative network and milia-like cysts, have no 7-point
counterpart at all.

And the ISIC 2018 Task 2 ground truth records **no qualifier**. A pixel is labelled
`pigment_network` or it is not; nothing says whether that network is typical or atypical. For
pigment network, streaks and globules the qualifier is *the entire diagnostic content* - typical
network is a benign sign, atypical network carries OR 2.8. A model that detects unqualified
"pigment network" has learned a diagnostically **ambiguous** concept.

The ISIC task page itself is silent on why these five: it calls them only "established
clinically-meaningful visual skin lesion patterns"
(<https://challenge.isic-archive.com/landing/2018/46/>) and gives no reference to the 7-point
checklist, pattern analysis or any other algorithm. Meanwhile the two *strongest* single
predictors in Carrera's study - marked architectural disorder (OR 6.6) and pattern asymmetry
(OR 4.9) - are not among the five, and the five that are included have ORs between 1.4 and 2.8
with poor interobserver agreement: atypical network **ICC 0.21**, negative network **ICC 0.15**,
which Carrera classes as "poor". The challenge organisers reach the same place from the other
direction, attributing Task 2's low scores in part to the fact that "dermoscopic attributes tend
to have poor inter-observer correlation among expert clinicians", citing Carrera
(<https://arxiv.org/abs/1902.03368>).

**Two consequences, both acted on below.**

1. The concept vector does not stop at presence. `n_blobs` and `asymmetry` exist precisely as an
   attempt to recover geometrically some of the regular/irregular qualifier that the ISIC labels
   throw away - "many scattered foci" and "asymmetrically distributed" are the observable side
   of *irregular*. **Whether that attempt works is unvalidated.** It is a plausible proxy, not a
   measured one, and it is the first thing a dermatologist reviewing this notebook should be
   asked about.
2. Every clinical sentence the rationale generator prints is framed as a *textbook association
   for the named structure*, never as a finding about the patient, and never with a polarity for
   the three ambiguous attributes. The wording is in the next cell so it can be audited in one
   place.

## Prior art, and a result that puts a ceiling on this

* **derm7pt** - Kawahara, Daneshvar, Argenziano, Hamarneh, *Seven-Point Checklist and Skin
  Lesion Classification Using Multitask Multimodal Neural Networks*, IEEE J Biomed Health Inform
  2019;23(2):538-546 - <https://ieeexplore.ieee.org/document/8333693/>, data at
  <https://derm.cs.sfu.ca>. 1,011 cases, paired clinical + dermoscopic images, and labels for
  **all seven checklist criteria with their qualifiers** (pigment network is annotated
  absent / typical / atypical). **This is the dataset this notebook should have been built on**,
  and the honest reason it was not is that this repo is an ISIC 2018 project. derm7pt has the
  qualifier and no masks; ISIC Task 2 has masks and no qualifier.
* **MONET** - Kim, Gadgil, DeGrave, Omiye, Cai, Daneshjou, Lee, *Transparent medical image AI
  via an image-text foundation model grounded in medical literature*, **Nature Medicine
  2024;30(4):1154-1165** - <https://www.nature.com/articles/s41591-024-02887-x>. (Frequently
  mis-cited as CVPR 2024; the citable publication is Nature Medicine.) An image-text model
  trained on 105,550 dermatology image-text pairs that scores images for concept presence, and
  annotates concepts "competitively with supervised models" - i.e. a concept bottleneck may not
  need hand-drawn masks at all. Directly relevant here: they audited the ISIC archive
  (>70,000 dermoscopic images) and found differences in how concepts correlate with benign
  versus malignant labels, which is concept-level confounding in this exact dataset family.
* **Concept bottlenecks on skin data already exist**: Patricio, Neves, Teixeira, *Coherent
  Concept-based Explanations in Medical Image and Its Application to Skin Lesion Diagnosis*,
  CVPRW 2023 - <https://arxiv.org/abs/2304.04579>; the same authors' vision-language variant
  over PH2, derm7pt and ISIC 2018 - <https://arxiv.org/abs/2311.14339>; and ExAID (Lucieri et
  al., Comput Methods Programs Biomed 2022;215:106620 - <https://arxiv.org/abs/2201.01249>),
  which builds text + visual explanations from clinician-defined dermoscopic concepts using
  concept activation vectors.
* **The ceiling result, and it is the most load-bearing citation in this notebook.** Napoles,
  Grau, Salgueiro, *Concept Inconsistency in Dermoscopic Concept Bottleneck Models: A Rough-Set
  Analysis of the Derm7pt Dataset* - <https://arxiv.org/html/2604.19323>, Sci Rep
  <https://www.nature.com/articles/s41598-026-56927-2>. Over derm7pt's seven criteria, 50 of
  305 unique concept profiles (16.4%) are **inconsistent**: identical concept vectors carry
  conflicting diagnoses. 30.3% of images, and **54.0% of all melanomas**, sit in rough-set
  boundary regions. They derive a **provable ceiling of 92.1% accuracy for any hard concept
  bottleneck using only those seven concepts**, "regardless of backbone architecture, training
  procedure, or regularization strategy". The two worst offenders are *irregular dots and
  globules* (present in 79% of boundary images) and *irregular streaks* (39%) - two of the five
  attributes used here.

  That result is about derm7pt's seven *qualified* concepts, not ISIC's five unqualified masks,
  so the 92.1% number does not transfer to this notebook. The direction of the argument does: a
  small set of dermoscopic concepts is provably insufficient to separate these diagnoses, and
  the shortfall is worst on melanoma. Whatever accuracy the cells below produce, the gap to a
  black box should be read as partly irreducible rather than as a tuning failure.


# Import library

In [ ]:
import os
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
import tensorflow as tf
from tensorflow.keras.layers import Dense, GlobalAveragePooling2D
from tensorflow.keras.models import Model
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import ModelCheckpoint, CSVLogger, ReduceLROnPlateau, EarlyStopping
import numpy as np
import pandas as pd
import random, math, cv2
from glob import glob
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier, export_text
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score,
                             roc_auc_score, confusion_matrix, classification_report)
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches

"""Trimmed on purpose: every name imported here is used below. The repo README's
own regret list flags `EfficientNetB0` sitting unused in task 3 as the tell that a
notebook is describing a plan rather than the code, so this one does not repeat it."""

# **Environment and data roots**

Resolved at runtime, printed, and checked - so a run documents its own inputs and fails with a specific message rather than a `KeyError` forty cells later. The same notebook has to work against the Colab/Drive layout tasks 1-3 use and against a Kaggle kernel with the datasets attached, where archive directories are nested twice (`.../X/X/`) and mirror slugs change, so nothing is a literal path.

In [ ]:
"""====================================================================
   ENVIRONMENT + DATA ROOT RESOLUTION
   This notebook has to run in two places: the Colab/Drive setup tasks
   1-3 were written for, and a Kaggle kernel with the datasets attached.
   Nothing below is a literal path - every root is globbed for and then
   printed, so a run documents its own inputs.
   ==================================================================="""
import sys, os
from glob import glob

ON_KAGGLE = os.path.isdir('/kaggle/input')
ON_COLAB  = 'google.colab' in sys.modules

def find_dir(patterns, what):
    """first existing directory matching any glob pattern, else None.
    Sorted so a run is deterministic when several mirrors are attached."""
    for pat in patterns:
        hits = sorted(d for d in glob(pat) if os.path.isdir(d))
        if hits:
            if len(hits) > 1:
                print(f'  note: {len(hits)} candidates for {what}, taking the first')
            return hits[0]
    return None

def find_file(patterns, what):
    for pat in patterns:
        hits = sorted(p for p in glob(pat) if os.path.isfile(p))
        if hits:
            if len(hits) > 1:
                print(f'  note: {len(hits)} candidates for {what}, taking the first')
            return hits[0]
    return None

if ON_KAGGLE:
    ENV = 'kaggle'
    """Kaggle mirrors nest the archive name twice (.../X/X/), and the mirror
    slugs change, so glob for the archive directory NAME anywhere under
    /kaggle/input rather than trusting a full path."""
    T2_GT_DIR      = find_dir(['/kaggle/input/**/ISIC2018_Task2_Training_GroundTruth_v3/ISIC2018_Task2_Training_GroundTruth_v3',
                               '/kaggle/input/**/ISIC2018_Task2_Training_GroundTruth_v3'],
                              'task-2 attribute GT')
    T12_IMG_DIRS   = [d for d in [
                        find_dir(['/kaggle/input/**/ISIC2018_Task1-2_Training_Input/ISIC2018_Task1-2_Training_Input',
                                  '/kaggle/input/**/ISIC2018_Task1-2_Training_Input'], 'task 1-2 input'),
                        find_dir(['/kaggle/input/**/ISIC2018_Task1-2_Validation_Input/ISIC2018_Task1-2_Validation_Input',
                                  '/kaggle/input/**/ISIC2018_Task1-2_Validation_Input'], 'task 1-2 val input'),
                      ] if d]
    T3_GT_CSVS     = [p for p in [
                        find_file(['/kaggle/input/**/ISIC2018_Task3_Training_GroundTruth/ISIC2018_Task3_Training_GroundTruth.csv',
                                   '/kaggle/input/**/ISIC2018_Task3_Training_GroundTruth.csv'], 'task-3 train csv'),
                        find_file(['/kaggle/input/**/ISIC2018_Task3_Validation_GroundTruth.csv'], 'task-3 val csv'),
                      ] if p]
    T3_IMG_DIRS    = [d for d in [
                        find_dir(['/kaggle/input/**/ISIC2018_Task3_Training_Input/ISIC2018_Task3_Training_Input',
                                  '/kaggle/input/**/ISIC2018_Task3_Training_Input',
                                  '/kaggle/input/**/HAM10000_images*'], 'task-3 images'),
                        find_dir(['/kaggle/input/**/ISIC2018_Task3_Validation_Input/ISIC2018_Task3_Validation_Input',
                                  '/kaggle/input/**/ISIC2018_Task3_Validation_Input'], 'task-3 val images'),
                      ] if d]
    T2_MODEL_CANDIDATES = ['/kaggle/input/**/u_net_task2.h5', '/kaggle/working/**/u_net_task2.h5']
    save_dir = '/kaggle/working/concept_bottleneck'

else:
    ENV = 'colab' if ON_COLAB else 'local'
    """the layout tasks 1-3 already use, documented in data/README.md"""
    ROOT = '/content/drive/MyDrive/ISIC2018'
    T2_GT_DIR      = find_dir([f'{ROOT}/Dataset/t2_train_masks/ISIC2018_Task2_Training_GroundTruth_v3',
                               f'{ROOT}/Dataset/**/ISIC2018_Task2_Training_GroundTruth_v3'],
                              'task-2 attribute GT')
    T12_IMG_DIRS   = [d for d in [
                        find_dir([f'{ROOT}/Dataset/t1_train_images/train_t12'], 'task 1-2 input'),
                        find_dir([f'{ROOT}/Dataset/t1_val_images/val_t12'], 'task 1-2 val input'),
                      ] if d]
    T3_GT_CSVS     = [p for p in [
                        find_file([f'{ROOT}/Dataset/Task23_gt/t3_train_gt/ISIC2018_Task3_Training_GroundTruth.csv'], 'task-3 train csv'),
                        find_file([f'{ROOT}/Dataset/Task23_gt/t3_val_gt/ISIC2018_Task3_Validation_GroundTruth.csv'], 'task-3 val csv'),
                      ] if p]
    T3_IMG_DIRS    = [d for d in [
                        find_dir([f'{ROOT}/Dataset/t3_train/train_t3'], 'task-3 images'),
                        find_dir([f'{ROOT}/Dataset/t3_val/val_t3'], 'task-3 val images'),
                      ] if d]
    T2_MODEL_CANDIDATES = [f'{ROOT}/Models/u_net_task2/u_net_task2.h5']
    save_dir = f'{ROOT}/Models/concept_bottleneck'

task2_model_path = find_file(T2_MODEL_CANDIDATES, 'trained task-2 model')

print('environment          :', ENV)
print('task-2 attribute GT  :', T2_GT_DIR)
print('task 1-2 image dirs  :', T12_IMG_DIRS)
print('task-3 label csv(s)  :', T3_GT_CSVS)
print('task-3 image dirs    :', T3_IMG_DIRS)
print('task-2 model weights :', task2_model_path)
print('outputs will go to   :', save_dir)

"""fail loudly and specifically, rather than 40 cells later with a KeyError"""
missing = []
if T2_GT_DIR is None:     missing.append('task-2 attribute ground truth directory')
if not T12_IMG_DIRS:      missing.append('task 1-2 input images')
if not T3_GT_CSVS:        missing.append('task-3 ground-truth csv')
if not T3_IMG_DIRS:       missing.append('task-3 input images (needed to train the label predictor)')
if task2_model_path is None:
    missing.append('trained task-2 u_net_task2.h5 (train task 2 first, or attach it as a dataset)')
if missing:
    print('\nMISSING INPUTS - the following cells that need them will not run:')
    for m in missing:
        print('  -', m)
else:
    print('\nall inputs resolved.')

# **Dataset paths and attribute definition**

`ATTRIBUTES` is copied verbatim from `task2-attribute-detection.ipynb` and is the single source of truth for channel order: channel 0 is always pigment network, channel 4 is always globules. **The order must never be reshuffled.** Nothing would raise - the task-2 `.h5` channels, the fitted thresholds, the 20 concept slots and every printed rationale are all indexed positionally off this list, so reordering it silently relabels every explanation the notebook produces.

In [ ]:
H, W = 256, 256

"""channel order of the (256, 256, 5) target - MUST NEVER BE RESHUFFLED.
Copied verbatim from task2-attribute-detection.ipynb. Everything downstream is
indexed positionally off this list: the task-2 .h5 output channels, the fitted
per-attribute thresholds, the 20 concept-vector slots, every logistic-regression
coefficient, and every sentence the rationale generator prints. Reordering it does
not raise an error anywhere - it silently relabels every explanation."""
ATTRIBUTES = [
    "pigment_network",
    "negative_network",
    "streaks",
    "milia_like_cyst",
    "globules",
]
N_ATTR = len(ATTRIBUTES)

"""short labels for tables/plots - same as task 2"""
ATTR_LABELS = {
    "pigment_network":  "Pigment network",
    "negative_network": "Negative network",
    "streaks":          "Streaks",
    "milia_like_cyst":  "Milia-like cysts",
    "globules":         "Globules",
}

"""one distinct colour per attribute, BGR because everything here goes through cv2.
Same colours as task 2 so the two notebooks' overlays read against each other."""
ATTR_COLORS_BGR = {
    "pigment_network":  (0, 0, 255),      # red
    "negative_network": (0, 255, 255),    # yellow
    "streaks":          (0, 255, 0),      # green
    "milia_like_cyst":  (255, 0, 255),    # magenta
    "globules":         (255, 128, 0),    # blue-cyan
}

"""task-3 class order - EXACTLY the column order of ISIC2018_Task3_*_GroundTruth.csv,
which is also the order task-3's `class_names` used. Asserted at load time."""
DIAGNOSES = ['MEL', 'NV', 'BCC', 'AKIEC', 'BKL', 'DF', 'VASC']
N_CLASS = len(DIAGNOSES)
DIAGNOSIS_LABELS = {
    'MEL':   'Melanoma',
    'NV':    'Melanocytic nevus',
    'BCC':   'Basal cell carcinoma',
    'AKIEC': 'Actinic keratosis / Bowen disease',
    'BKL':   'Benign keratosis-like lesion',
    'DF':    'Dermatofibroma',
    'VASC':  'Vascular lesion',
}

"""which of the 7 classes are melanocytic, i.e. the ones the 5 attributes were
defined to describe at all. Used only to caveat the results, never as a feature."""
MELANOCYTIC = {'MEL', 'NV'}

def create_dir(path):
    if not os.path.exists(path):
        os.makedirs(path)

create_dir(save_dir)

# **The clinical wording, in one auditable place**

Every clinical sentence the rationale generator can print lives in this cell, so it can be handed to a dermatologist as a single object. It is a non-clinician's paraphrase of the literature above and has not been reviewed by anyone qualified.

In [ ]:
"""====================================================================
   The exact clinical wording the rationale generator is allowed to use.
   Kept in ONE place so it can be handed to a dermatologist as a single
   reviewable object rather than hunted through f-strings.

   Rules this wording follows:
    * it describes a textbook association for the STRUCTURE, never a
      finding about the patient;
    * for the three attributes whose meaning depends on a qualifier ISIC
      does not record, it states the ambiguity instead of picking a side;
    * it carries its own source.
   NOTHING HERE HAS BEEN REVIEWED BY A CLINICIAN. It is a literature
   paraphrase written by a non-clinician and must be treated as such.
   ==================================================================="""

CLINICAL_NOTE = {
    "pigment_network":
        "core pattern-analysis criterion, but the direction depends on TYPICAL vs "
        "ATYPICAL, which the ISIC mask does not record. Typical networks are common in "
        "benign nevi; atypical networks carry OR 2.8 (2.4-3.4) for melanoma "
        "(Carrera 2016). This detection is therefore diagnostically AMBIGUOUS.",
    "negative_network":
        "melanoma-leaning but weak and insensitive, and NOT one of the 7-point "
        "criteria: sensitivity 34.6%, specificity ~77%, OR 1.8 (1.3-2.7); also present "
        "in 28.8% of Spitz nevi and 18.2% of ordinary nevi (Pizzichetta 2013).",
    "streaks":
        "7-point criterion #4 when IRREGULAR (1 point). Distribution decides: streaks "
        "spread symmetrically around the whole lesion favour a benign Reed/Spitz nevus, "
        "asymmetric distribution is the melanoma-suspicious pattern. ISIC does not "
        "record which. Streaks OR 1.5 (1.3-1.8), pseudopods OR 2.1 (Carrera 2016).",
    "milia_like_cyst":
        "NOT a melanoma criterion in any of the standard algorithms; a keratinocytic "
        "clue classically associated with seborrheic keratosis. Only the CLOUDY subtype "
        "carries the high specificity (99.1% for SK vs melanoma); STARRY cysts occur in "
        "melanoma too, and they also appear in congenital-pattern nevi. ISIC does not "
        "record the subtype, so this must NOT be read as evidence of benignity.",
    "globules":
        "7-point criterion #5 when IRREGULAR (1 point). Uniform, evenly distributed "
        "globules are the benign pattern; varying size/shape/colour and focal peripheral "
        "distribution is the melanoma pattern. ISIC does not record which. Irregular "
        "black dots OR 1.8, irregular black globules OR 1.9 (Carrera 2016).",
}

"""does the attribute map onto a 7-point checklist criterion at all?"""
SEVEN_POINT = {
    "pigment_network":  "#1 atypical pigment network - MAJOR, 2 points (only if atypical)",
    "negative_network": "not a 7-point criterion",
    "streaks":          "#4 irregular streaks - minor, 1 point (only if irregular)",
    "milia_like_cyst":  "not a 7-point criterion",
    "globules":         "#5 irregular dots/globules - minor, 1 point (only if irregular)",
}

"""whether presence alone has a defensible direction. Used only to caveat text -
NEVER as a feature, a weight, or a prior. The model is not told any of this."""
POLARITY = {
    "pigment_network":  "AMBIGUOUS (typical benign / atypical suspicious)",
    "negative_network": "weakly melanoma-leaning",
    "streaks":          "AMBIGUOUS (symmetric benign / asymmetric suspicious)",
    "milia_like_cyst":  "leans seborrheic keratosis, subtype-dependent, NOT proof of benign",
    "globules":         "AMBIGUOUS (regular benign / irregular suspicious)",
}

for a in ATTRIBUTES:
    print(f"{ATTR_LABELS[a]}")
    print(f"   7-point : {SEVEN_POINT[a]}")
    print(f"   polarity: {POLARITY[a]}")
    print(f"   note    : {CLINICAL_NOTE[a]}")
    print()

n_ambig = sum(1 for a in ATTRIBUTES if POLARITY[a].startswith('AMBIGUOUS'))
print(f'{n_ambig} of {N_ATTR} attributes have NO defensible direction from presence alone,')
print('and 2 of 5 are not 7-point criteria at all. That is the honest starting position.')

# **Create dataset helpers and metrics**

Preprocessing is byte-identical to task 1 and task 2 - same resize, same `/255.0`, same BGR-from-`cv2` - because `g`'s weights were fitted on exactly that and any 'improvement' here would silently degrade every concept. Masks use `INTER_NEAREST` and `> 127` for the same reason task 2 does: bilinear puts grey fringes on these thin structures and erases the smallest cysts outright.

In [ ]:
def read_image(path):
    """(256, 256, 3) float32 in [0,1] - byte-identical preprocessing to task 1 and
    task 2, because the task-2 weights were fitted on exactly this. cv2 loads BGR
    and task 1/2 never converted, so g() expects BGR; do not "fix" that here."""
    x = cv2.imread(path, cv2.IMREAD_COLOR)
    if x is None:
        return None
    x = cv2.resize(x, (W, H))
    x = x / 255.0
    return x.astype(np.float32)

def read_attribute_masks(paths):
    """(256, 256, 5) binary, ATTRIBUTES channel order.
    Missing / unreadable file -> all-zero channel, which is the correct semantics:
    that attribute is absent. INTER_NEAREST + >127, never bilinear - bilinear puts
    grey fringes on these thin structures and deletes the smallest cysts outright."""
    out = np.zeros((H, W, N_ATTR), dtype=np.float32)
    for c, p in enumerate(paths):
        if p == "" or not os.path.exists(p):
            continue
        m = cv2.imread(p, cv2.IMREAD_GRAYSCALE)
        if m is None:
            continue
        m = cv2.resize(m, (W, H), interpolation=cv2.INTER_NEAREST)
        out[..., c] = (m > 127).astype(np.float32)
    return out

def attribute_paths(image_id, gt_dir):
    """the 5 ground-truth paths for one image id, "" when the file does not exist"""
    return [os.path.join(gt_dir, f"{image_id}_attribute_{a}.png")
            if os.path.exists(os.path.join(gt_dir, f"{image_id}_attribute_{a}.png")) else ""
            for a in ATTRIBUTES]

def image_ids_in(dirs):
    """{ISIC_0000000: /abs/path.jpg} merged over a list of directories"""
    out = {}
    for d in dirs:
        for p in sorted(glob(os.path.join(d, "*.jpg"))) + sorted(glob(os.path.join(d, "*.JPG"))):
            out[os.path.basename(p).rsplit('.', 1)[0]] = p
    return out

def attribute_gt_ids(gt_dir):
    """{ISIC_0000000: [5 mask paths]} for every id that has at least one mask FILE.

    NOTE, and this cost me a wrong assumption: in the ISIC 2018 Task 2 ground truth
    EVERY image ships ALL FIVE mask files, and absence of an attribute is encoded as
    an ALL-ZERO PNG, not as a missing file. So "the file exists" tells you nothing
    about whether the attribute is present - that has to be read from the pixels
    (see attribute_presence_stats below). Both facts are printed rather than assumed."""
    ids = {}
    for p in sorted(glob(os.path.join(gt_dir, "*_attribute_*.png"))):
        base = os.path.basename(p)
        image_id = base.split('_attribute_')[0]
        ids.setdefault(image_id, None)
    for image_id in list(ids):
        ids[image_id] = attribute_paths(image_id, gt_dir)
    return ids

def load_task3_labels(csv_path):
    """{ISIC_0000000: class_index} in DIAGNOSES order.
    The csv is one-hot over 7 columns. argmax is only safe if the columns are in
    the DIAGNOSES order, so that is asserted rather than trusted."""
    df = pd.read_csv(csv_path)
    cols = [c for c in df.columns if c.lower() != 'image']
    assert cols == DIAGNOSES, f"unexpected csv column order {cols}, expected {DIAGNOSES}"
    onehot = df[DIAGNOSES].to_numpy().astype(float)
    ids = df[df.columns[0]].astype(str).to_numpy()
    return dict(zip(ids, onehot.argmax(axis=1).astype(int)))

In [ ]:
smooth = 1e-15

"""task-1 / task-2 metric idiom, kept identical so the three notebooks stay comparable"""
def iou(y_true, y_pred):
    def f(y_true, y_pred):
        intersection = (y_true * y_pred).sum()
        union = y_true.sum() + y_pred.sum() - intersection
        x = (intersection + 1e-15) / (union + 1e-15)
        x = x.astype(np.float32)
        return x
    return tf.numpy_function(f, [y_true, y_pred], tf.float32)

def dice_coef(y_true, y_pred):
    y_true = tf.keras.layers.Flatten()(y_true)
    y_pred = tf.keras.layers.Flatten()(y_pred)
    intersection = tf.reduce_sum(y_true * y_pred)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_true) + tf.reduce_sum(y_pred) + smooth)

def dice_loss(y_true, y_pred):
    return 1.0 - dice_coef(y_true, y_pred)

def dice_coef_channel(y_true, y_pred, c):
    y_t = tf.reshape(y_true[..., c], [-1])
    y_p = tf.reshape(y_pred[..., c], [-1])
    intersection = tf.reduce_sum(y_t * y_p)
    return (2. * intersection + smooth) / (tf.reduce_sum(y_t) + tf.reduce_sum(y_p) + smooth)

def dice_macro(y_true, y_pred):
    return tf.add_n([dice_coef_channel(y_true, y_pred, c) for c in range(N_ATTR)]) / float(N_ATTR)

def dice_loss_multilabel(y_true, y_pred):
    return 1.0 - dice_macro(y_true, y_pred)

"""numpy pooled Jaccard, used for the concept-fidelity check further down.
Pooled over the whole subset, not per image - same reasoning as task 2:
most images have no positive pixels for most attributes."""
def pooled_jaccard(y_true_flat, y_pred_flat):
    inter = float((y_true_flat * y_pred_flat).sum())
    union = float(y_true_flat.sum() + y_pred_flat.sum() - inter)
    return inter / union if union > 0 else np.nan

# **The data join, which is the actual problem**

A concept bottleneck model needs, in the general case, images carrying **both** a concept target
and a label target. This repo's two halves are:

* **Task 1-2 training input** - the images with attribute masks. The ISIC 2018 challenge paper
  reports **2,594 training images with 12,970 ground-truth masks** (5 per image) for Task 2
  (Codella et al., <https://arxiv.org/abs/1902.03368>).
* **Task 3 training input** - HAM10000, **10,015** images with a 7-class diagnosis
  (<https://arxiv.org/abs/1803.10417>).

Both are ISIC archive images with `ISIC_xxxxxxx` ids. So: how much do they overlap?

## The answer: exactly zero, and this is settled

Two independent measurements, neither of which relies on anyone's description of the data.

**1. The official distribution files.** Reading the ZIP central directories straight off
`isic-challenge-data.s3.amazonaws.com/2018/`:

| source | unique ids | id range |
|---|---|---|
| `ISIC2018_Task1-2_Training_Input.zip` | **2,594** | `ISIC_0000000` - `ISIC_0016072` |
| `ISIC2018_Task1_Training_GroundTruth.zip` | 2,594 (identical set) | `ISIC_0000000` - `ISIC_0016072` |
| `ISIC2018_Task3_Training_GroundTruth.csv` | **10,015** | `ISIC_0024306` - `ISIC_0034320` |

**Intersection = 0.** The ranges do not merely fail to overlap, they do not come close: Task 1-2
training stops at `ISIC_0016072` and HAM10000 starts at `ISIC_0024306`, roughly 8,000 ids later.

**2. The ISIC Archive API.** The archive's collections reproduce the challenge splits exactly
(sizes match the downloads: 2594 / 100 / 1000 for Task 1-2 train/val/test, 10015 / 193 / 1512 for
Task 3). Every pairwise intersection of a Task 1-2 collection with any HAM10000 collection is 0.
The check that proves the machinery is not simply broken: Task 3 train + val + test = 10,015 +
193 + 1,512 = **11,720**, which is exactly the archive's full HAM10000 collection, and Task 3
train is a strict subset of it. So the zero is real, not a failed filter.

## Why - the provenance is entirely different

This is not a coincidence of numbering. Every metadata axis separates the two sets:

| | Task 1-2 Training (2,594) | Task 3 Training (10,015) |
|---|---|---|
| license, 100% of images | **CC-0** | **CC-BY-NC** |
| attribution, 100% | **"Anonymous"** | **"MILK study team"**, ViDIR Group, Medical University of Vienna |
| image resolutions | **206 distinct**, up to 4288x2848 | **one**: every image 600x450 |
| diagnoses present | only nevus / melanoma / seborrheic keratosis - the **ISIC 2017** class set | all 7 HAM classes, incl. BCC, AKIEC, DF, VASC |
| archive origin | legacy **MSK-1...5 / UDA-1,2 / SONIC**; ~94% are ISIC **2017** challenge images | HAM10000 |

The `ATTRIBUTION.txt` inside the official archives says it outright - Task 1-2:
*"ISIC-2018 Challenge: Task 1-2: Training Input by Anonymous ... CC0"*; Task 3:
*"ISIC-2018 Challenge: Task 3: Training Ground Truth (HAM10000 Dataset ...) (c) by ViDIR Group,
Department of Dermatology, Medical University of Vienna ... CC BY-NC"*.

So **the Task 1-2 training set is essentially the recycled ISIC 2016/2017 legacy-archive subset**,
not HAM10000 at all.

The clearest published confirmation is Mirikharaji et al., *A Survey on Deep Learning for Skin
Lesion Segmentation*, Medical Image Analysis - <https://arxiv.org/abs/2206.00356>:

> "**ISIC 2018 provided, for the first time, separate datasets for the tasks**, with 2,594
> training (20% melanomas, 72% nevi, and 8% seborrheic keratoses) ... for the tasks of
> segmentation and feature extraction, and 10,015/1,512 training/test images for the
> classification task, all with 600x450 pixels. The training dataset for classification was the
> HAM10000 dataset..."

Their 20 / 72 / 8 split matches the archive metadata exactly. Note that the challenge paper
itself (Codella et al., <https://arxiv.org/abs/1902.03368>) is **silent** on Task 1-2 provenance -
it cites HAM10000 for Part 3 and cites no data source at all for the 2,594 images, which is
presumably why the question keeps getting asked.

## The claim that there *is* overlap, and why you will meet it

Searching this question returns, repeatedly, one unsourced sentence. From
<https://arxiv.org/abs/2510.17773>, verbatim: *"The images are similar in content to HAM10000
(and indeed, many HAM10000 images are included in this challenge)"*. That parenthetical is
**false** - 0 of 2,594 - and it propagates through Kaggle dataset descriptions.

Its likely origin is a half-true remark from ISIC staff on the archive forum
(<https://forum.isic-archive.com/t/isic-2018-lesion-class-for-tasks-1-and-2/1391>), answering
exactly this question: the Task 1-2 **test** images *"are derived from the same source population
as the images in HAM10000, and thus the distribution of disease class is comparable."* That is
accurate as far as it goes, and it does not go where people take it. About 30% of Task 1-2
val/test **is** attributed to the same Vienna department as HAM10000 - but those are newly
accessioned images with ids *above* HAM10000's ceiling, none are 600x450, and the id intersection
is still exactly 0. And the Task 1-2 **training** set contains *zero* Vienna-attributed images, so
the remark does not extend to it at all.

"Same source population" is not "same images". The runtime cell below recomputes everything and
prints the numeric id ranges, so a future mirror that differs will say so rather than let this
notebook inherit a stale assumption.

## Three caveats on the counts

1. **A mirror may not be the official split.** A probe of the Kaggle mirror of the attribute
   ground truth found **3,694** ids and 18,470 mask files, against the official **2,594**. Note
   that 2,594 + 100 + 1,000 = **3,694** exactly - the Task 1-2 train, validation and test splits
   concatenated. That arithmetic is suggestive rather than proven, and it matters: if the
   directory bundles all three splits, then a "held-out" subset drawn from it will contain images
   `g` was trained on, and the concept-fidelity numbers below are optimistic by an unknown amount.
   The fidelity cell says so and prints the id count it actually used.
2. **Every id ships all five mask files.** Absence of an attribute is encoded as an *all-zero
   PNG*, not a missing file, so `os.path.exists` tells you nothing about presence - that has to be
   read from the pixels. The prevalence cell further down does exactly that.
3. **Identity was checked by id, not by pixels.** Perceptual near-duplicates under different ISIC
   ids have not been formally excluded. Cassidy et al. (<https://www.sciencedirect.com/science/article/pii/S1361841521003509>)
   did hash/SSIM duplicate detection across ISIC 2016-2020 but did not publish this particular
   comparison. Given different institutions, licenses and acquisition resolutions, hidden
   duplication is implausible - but it is unproven, so this is honest uncertainty rather than a
   guarantee.

## What an empty intersection does to the design

Of the three CBM regimes, two need `c` and `y` on the same image:

| regime | needs `c_true` + `y` together? | available here |
|---|---|---|
| Independent | yes - `f` is fitted on true concepts | **no** |
| Joint | yes - one gradient step consumes both targets | **no** |
| **Sequential** | **no** - `f` is fitted on `g(x)`, so it only needs labels | **yes** |

So for the 7-class task, sequential is not a compromise choice; it is the only reachable one.

**But the gate can be opened, and it is worth opening.** The Task 1-2 images *do* have diagnoses -
just not in the Task 3 csv. All 2,594 carry `diagnosis` metadata in the ISIC Archive
(`api.isic-archive.com`, collection 63), and the notebook fetches them as an opt-in step. The
catch is the class set: those legacy images have only **three** diagnoses - melanoma, nevus,
seborrheic keratosis - with no BCC, AKIEC, DF or VASC anywhere. And the mapping is not clean:
HAM10000's `BKL` is *broader* than seborrheic keratosis (it also covers solar lentigines and
lichen-planus-like keratoses), so SK -> BKL is a subset relation, not an equality.

That is not a replacement for the 7-class pipeline. It is something better than nothing and
narrower than the real thing, so this notebook builds it as a separate, clearly-labelled
**audit bench**:

| | main pipeline (Route B) | audit bench (Route A') |
|---|---|---|
| images | Task 3 / HAM10000 | Task 1-2 |
| classes | **7** | **3** (MEL / NV / SK-as-BKL) |
| concepts | predicted only | predicted **and** ground truth |
| regimes | sequential | **independent, sequential, joint** |
| concept fidelity on the classified images | impossible | **yes** |
| real Koh intervention | impossible | **yes** |
| per-image concept-vs-reasoning attribution | impossible | **yes** |
| honest to quote as headline accuracy | yes | **no** - 3 classes, and `g` trained on these images |

The bench has one loud caveat that is repeated wherever its numbers appear: **`g` was trained on
most of these images.** The task-2 U-Net saw the Task 1-2 training split, so fidelity and
intervention measured on the bench show `g` at its absolute best, on its own training
distribution. They are an upper bound on how well the concept extractor can behave - never an
estimate of what it does on Route B.

**And this is closer to deployment than the alternative.** The fallback - run `g` inference-only
over the Task 3 images to get *predicted* concept vectors, then fit `f` on those - is not a
degraded substitute for the "real" thing. It is what any deployed system does. At inference time
nobody has attribute annotations for the lesion in front of them; the only concepts available are
the ones the model predicts. Route B trains `f` on exactly the distribution it will see in use,
which is the sequential regime's actual argument for existing.

What it genuinely costs, and none of this is recoverable by being clever:

1. **The concepts are model-derived, not ground-truth-supervised, on every image `f` ever sees.**
   Nothing anchors bottleneck dimension `k` to its clinical name except training that happened on
   a *different set of images*.
2. **Concept fidelity cannot be measured where it matters.** It is measurable only on held-out
   Task 2 images (`ISIC_00xxxxx`) and then *assumed* to carry over to Task 3 images
   (`ISIC_002xxxx`-`ISIC_003xxxx`) - a different acquisition batch, different equipment,
   different centres. That assumption is the single largest unverified step in this notebook and
   it is restated wherever a fidelity number appears.
3. **On the 7-class task the Koh et al. intervention experiment is impossible.** Overwriting a
   predicted concept with its ground-truth value requires ground truth on the classified image,
   and there is none. It is run on the audit bench instead, where it is real; the 7-class pipeline
   gets a clearly-labelled *synthetic* sensitivity analysis, which answers the strictly weaker
   question "how much does `f` lean on this concept" and can report no accuracy gain at all,
   because there is nothing true to move towards.
4. **Concept errors and reasoning errors cannot be separated per-image on Route B.** Clean
   attribution runs on the bench; for the 7-class pipeline the cell degrades into a risk *triage*
   that multiplies fidelity (measured on one image set) by leverage (measured on another). That is
   a weaker instrument and it is labelled as one.

## Two other things this research turned up, both usable

* **Lesion masks for all of HAM10000 exist**, released separately by Tschandl after exactly this
  complaint: `HAM10000_segmentations_lesion_tschandl.zip`, 10,015 masks, Harvard Dataverse
  <https://doi.org/10.7910/DVN/DBW86T>. That matters here for a specific reason - `area_frac` is
  normalised by the lesion area when a lesion mask is available and by the whole frame otherwise,
  and only the former is comparable across images shot at different magnifications. So this
  dataset makes the *better* version of the concept vector reachable on Route B, without needing
  Task 1's model at all. Attach it and pass `lesion_model` (or the masks directly) to
  `predict_concepts`.
* A second, lower-quality mask collection covers all 11,720 HAM images (ISIC Archive collection
  531, <https://doi.org/10.34970/387951>) but is U-Net-generated with known centre bias, so the
  Tschandl release is the one to use.

Neither provides *attribute* masks, so neither fixes the concept-supervision problem. They fix
the normalisation problem only.


In [ ]:
"""====================================================================
   THE JOIN, MEASURED. Nothing here is hard-coded; every count comes
   from whatever is actually mounted. If a future mirror of the dataset
   changes, this cell changes its answer and the route below follows.
   ==================================================================="""

"""1. the concept side: every id with task-2 attribute ground truth"""
attr_gt = attribute_gt_ids(T2_GT_DIR) if T2_GT_DIR else {}
t12_images = image_ids_in(T12_IMG_DIRS)
n_mask_files = len(glob(os.path.join(T2_GT_DIR, "*_attribute_*.png"))) if T2_GT_DIR else 0

"""how many of the 5 files each id actually has - the check that revealed that
'file exists' does not mean 'attribute present'"""
files_per_id = np.array([sum(1 for p in v if p != "") for v in attr_gt.values()]) if attr_gt else np.array([])

print('task-2 attribute GT ids            :', len(attr_gt))
print('task-2 attribute mask FILES        :', n_mask_files)
if len(files_per_id):
    print('mask files per id (min/median/max) :',
          int(files_per_id.min()), int(np.median(files_per_id)), int(files_per_id.max()))
    print('ids with all 5 files present       :', int((files_per_id == N_ATTR).sum()))
print('task 1-2 input images on disk      :', len(t12_images))
print('  ...ids with GT but no image file :', len(set(attr_gt) - set(t12_images)))

"""2. the label side: every id with a task-3 diagnosis"""
t3_labels = {}
for csv_path in T3_GT_CSVS:
    t3_labels.update(load_task3_labels(csv_path))
t3_images = image_ids_in(T3_IMG_DIRS)
print('\ntask-3 labelled ids in the csv(s)  :', len(t3_labels))
print('task-3 input images on disk        :', len(t3_images))

"""3. the intersection - the set the CBM would ideally be trained on"""
JOIN_IDS = sorted(set(attr_gt) & set(t3_labels))
print('\n>>> |attribute GT AND diagnosis label| =', len(JOIN_IDS))

"""4. WHY, if it is empty: are the two id sets even in the same numeric range?
Descriptive, but in this case decisive - see the markdown above."""
def id_range(ids):
    nums = [int(i.split('_')[1]) for i in ids
            if i.startswith('ISIC_') and i.split('_')[1].isdigit()]
    return (f"ISIC_{min(nums):07d}", f"ISIC_{max(nums):07d}", len(nums)) if nums else (None, None, 0)

print('\nnumeric id range, attribute-GT set :', id_range(attr_gt))
print('numeric id range, task-3 label set :', id_range(t3_labels))

In [ ]:
"""per-class breakdown of the intersection. A total is not enough: a 7-class label
predictor needs every class populated, and a class with 0 or 1 member cannot be
learned or stratified however large the total is."""
if len(JOIN_IDS) > 0:
    y_join = np.array([t3_labels[i] for i in JOIN_IDS], dtype=int)
    counts = np.bincount(y_join, minlength=N_CLASS)
    breakdown = pd.DataFrame({
        'code':  DIAGNOSES,
        'class': [DIAGNOSIS_LABELS[c] for c in DIAGNOSES],
        'n in intersection': counts,
        '% of intersection': 100.0 * counts / len(JOIN_IDS),
    })
    print(breakdown.to_string(index=False))
    print('\nclasses with zero members  :',
          [DIAGNOSES[k] for k in range(N_CLASS) if counts[k] == 0])
    print('classes with fewer than 10 :',
          [DIAGNOSES[k] for k in range(N_CLASS) if 0 < counts[k] < 10])
else:
    y_join = np.array([], dtype=int)
    counts = np.zeros(N_CLASS, dtype=int)
    print('The intersection is EMPTY, so there is no per-class breakdown to print.')
    print()
    print('This is not a missing-file problem and not a bug in the cell above.')
    print('The two id sets are disjoint batches of the ISIC archive - see the')
    print('numeric ranges printed above. Everything below therefore takes the')
    print('PREDICTED-CONCEPT route, which is the deployment setting anyway.')

In [ ]:
"""====================================================================
   ROUTE SELECTION. Both branches are implemented. Which one runs is
   decided by the measured intersection, never by an assumption.
   ==================================================================="""
MIN_PER_CLASS  = 10     # below this a class cannot support a stratified split
MIN_JOIN_TOTAL = 200    # below this, 20 features x 7 classes is not trainable

n_usable_classes = int((counts >= MIN_PER_CLASS).sum())

if len(JOIN_IDS) >= MIN_JOIN_TOTAL and n_usable_classes >= 2:
    JOIN_MODE = 'gt_concepts'
    print('ROUTE A -- the intersection is large enough to supervise concepts directly.')
    print('  f() trains on GROUND-TRUTH concept vectors from the task-2 masks and is')
    print('  evaluated on PREDICTED ones. All three CBM regimes are available, and')
    print('  concept fidelity and GT concept intervention are both measurable on the')
    print('  same images the classifier is scored on.')
else:
    JOIN_MODE = 'predicted_concepts'
    print('ROUTE B -- no usable intersection (n =', len(JOIN_IDS),
          ', usable classes =', n_usable_classes, ').')
    print('  g() is run INFERENCE-ONLY over the task-3 images to produce PREDICTED')
    print('  concept vectors; f() is then trained on those against the task-3 labels.')
    print('  No attribute ground truth is needed on the task-3 side, which is the')
    print('  only reason this pipeline is buildable at all.')
    print()
    print('  What this costs, stated plainly:')
    print('   1. The concepts are MODEL-DERIVED, not ground-truth-supervised. Nothing')
    print('      anchors dimension k of the bottleneck to its clinical name except the')
    print('      task-2 training that happened on a DIFFERENT set of images.')
    print('   2. Concept errors propagate straight into the diagnosis and are')
    print('      indistinguishable, on these images, from reasoning errors.')
    print('   3. Concept fidelity CANNOT be measured on any image the label predictor')
    print('      sees, ever. It is measurable only on the task-2 held-out split, and')
    print('      that is a transfer assumption, not a measurement.')
    print('   4. INDEPENDENT and JOINT both need c_true paired with y on the same')
    print('      image. On this route only SEQUENTIAL is trainable.')
    print('   5. GT concept intervention is impossible on these images. For the')
    print('      7-class pipeline it is replaced by a clearly-labelled SYNTHETIC')
    print('      sensitivity analysis, which answers a weaker question.')
    print()
    print('  Points 2, 4 and 5 are recoverable on a NARROWER dataset. The task 1-2')
    print('  images do have diagnoses - in the ISIC Archive metadata, not in the')
    print('  task-3 csv - so they can carry attribute masks AND a label at once.')
    print('  That is the AUDIT BENCH built further down: 3 classes only (melanoma /')
    print('  nevus / seborrheic keratosis), and g() was trained on most of it, so it')
    print('  is a test rig for the architecture and NOT an accuracy claim.')

print('\nJOIN_MODE =', JOIN_MODE)

# **How often is each attribute actually present?**

Read from **pixel content**, not from whether a file exists - because every id in the Task 2 ground
truth ships all five mask files and absence is encoded as an all-zero PNG. This is the cell that
establishes that fact rather than assuming it either way.

The numbers are also the prevalence the presence threshold has to cope with, and they are worth
comparing against the published training-set frequencies as a sanity check that the mirror on disk
is the dataset it claims to be. Two sources, which agree on the ordering and differ slightly on the
values - worth knowing before treating either as exact:

* the challenge paper (<https://arxiv.org/abs/1902.03368>): pigment network **58.7%**,
  milia-like cysts 26.3%, globules 23.2%, negative network 7.3%, streaks **2.9%**;
* Le et al. (<https://arxiv.org/abs/2104.01641>), which is the source the repo README quotes:
  pigment network 58.7%, milia-like cysts 26.2%, globules 24.4%, negative network 7.5%,
  streaks 3.9%.

Either way streaks and negative network are single-digit-percent attributes, which is why `g` is
weakest exactly where the 7-point checklist would most like it to be strong: *irregular streaks* is
a scored criterion, and it is the rarest thing in the training data.


In [ ]:
"""How often is each attribute actually PRESENT in the task-2 ground truth?
Read from pixel content, because every id ships all five files and absence is an
all-zero PNG. This also gives the prevalence the presence-thresholds below have to
cope with, and it is the number to compare against the published training-set
frequencies (Le et al. 2022, arXiv:2104.01641) as a sanity check on the mirror."""
def attribute_presence_stats(ids, sample=None):
    keys = sorted(ids)
    if sample is not None and sample < len(keys):
        rng = np.random.RandomState(42)
        keys = [keys[i] for i in sorted(rng.choice(len(keys), size=sample, replace=False))]

    n_present = np.zeros(N_ATTR, dtype=np.int64)
    pos_pixels = np.zeros(N_ATTR, dtype=np.float64)
    total_pixels = 0.0
    for image_id in tqdm(keys, total=len(keys)):
        y = read_attribute_masks(ids[image_id])
        per_ch = y.reshape(-1, N_ATTR).sum(0)
        n_present += (per_ch > 0).astype(np.int64)
        pos_pixels += per_ch
        total_pixels += H * W
    return pd.DataFrame({
        'Attribute': [ATTR_LABELS[a] for a in ATTRIBUTES],
        'images WITH attribute': n_present,
        'present %': 100.0 * n_present / max(len(keys), 1),
        'positive pixel %': 100.0 * pos_pixels / max(total_pixels, 1),
    }), len(keys)

if attr_gt:
    presence_df, n_scanned = attribute_presence_stats(attr_gt)
    print('scanned', n_scanned, 'ids')
    print(presence_df.to_string(index=False))
else:
    presence_df = None
    print('no attribute ground truth resolved - skipped.')

# **Designing the bottleneck**

`g` outputs five binary masks. `f` needs a vector. The choice of what to put in that vector is
where a concept bottleneck is either interpretable or merely narrow.

**Not just presence.** The clinical section above establishes the reason: for pigment network,
streaks and globules the diagnostic content is almost entirely in the *qualifier* - typical vs
atypical, symmetric vs asymmetric, regular vs irregular - and ISIC's masks record no qualifier.
The only route to any of that is geometry. So each attribute contributes **four** scalars:

| feature | what it is | why a dermoscopy checklist would care |
|---|---|---|
| `present` | 1 if the attribute covers at least `PRESENCE_AREA_FRAC` of the reference area | The 7-point checklist and every algorithm like it score criteria as present/absent. This is also the **least leaky** dimension: a thresholded binary carries far less of the image than a continuous score (Mahinpei et al.; Havasi et al.'s hard-concept models leak least) |
| `area_frac` | positive area / reference area | Extent, not just presence. A pigment network across the whole lesion reads differently from a patch in one corner. **Most leaky of the four** - it is a continuous score, so treat coefficients on it with the most suspicion |
| `n_blobs` | connected components above `MIN_BLOB_PX` | Multifocality. Globules and milia-like cysts are inherently multi-focus; "few large" vs "many scattered" is a distinction clinicians draw, and "unevenly distributed" is part of the *irregular* globule pattern |
| `asymmetry` | `1 - IoU(mask, mask mirrored about its own centroid axis)` | The closest available proxy for the missing qualifier. Streaks distributed *symmetrically* around a lesion favour a benign Reed/Spitz nevus while *asymmetric* distribution is the melanoma-suspicious pattern; asymmetry of structure distribution is also what Carrera et al. found to be among the most *reliable* dermoscopic judgements (pattern asymmetry OR 4.9), unlike the individual structures |

5 attributes x 4 features = **20 dimensions**, all named, all printable, all with a one-line
clinical reading. That is the bottleneck. Nothing else reaches `f`.

**Two design decisions worth stating out loud.**

*The reference area.* `area_frac` divides by the lesion area when a Task-1 lesion mask is
supplied, and by the whole frame otherwise. Dividing by the lesion is better - it makes the
number comparable across images shot at different magnifications - and the function takes an
optional `lesion_model` for exactly that, chaining Task 1 in as well. When no lesion mask is
given the fallback is the frame, and that comparability is simply lost. The code says which it
did; the number means different things in the two cases.

*Where a side channel would go, and why there isn't one.* The obvious way to raise accuracy is to
concatenate some features of the raw image onto this vector - a hybrid or residual CBM. **That is
deliberately not done.** Everything downstream that makes this notebook worth writing - the
per-concept audit, the sensitivity analysis, the risk triage - depends on there being no path from
image to diagnosis that bypasses a named concept. PCBM's authors put it plainly about their own
residual variant: "in PCBM, we can remove the concept from the model, but in PCBM-h, there may
still be leftover information about the spurious concept in the residual part of the model."
If a future version adds a side channel, the interventional cells below stop meaning what they
say and must be deleted rather than re-run.

**Keep it small.** 20 dimensions for 7 classes is already generous: a multinomial logistic
regression fits 7 x 20 + 7 = 147 parameters, which is about the largest model whose coefficient
matrix can still be printed and read in one go. Every extra dimension is one more row of table a
human has to hold in their head, which is the actual budget being spent here.


In [ ]:
"""====================================================================
   THE BOTTLENECK. 4 named scalars x 5 attributes = 20 dimensions.
   Every dimension has a name and a one-line clinical reading.
   ==================================================================="""
MIN_BLOB_PX = 8          # components smaller than this are annotation/threshold speckle
PRESENCE_AREA_FRAC = 1e-4  # presence needs a minimum footprint, not one stray pixel

CONCEPT_SUFFIXES = ['present', 'area_frac', 'n_blobs', 'asymmetry']

CONCEPT_NAMES = [f"{a}__{s}" for a in ATTRIBUTES for s in CONCEPT_SUFFIXES]
N_CONCEPT = len(CONCEPT_NAMES)

CONCEPT_PRETTY = {
    f"{a}__{s}": f"{ATTR_LABELS[a]} - {p}"
    for a in ATTRIBUTES
    for s, p in zip(CONCEPT_SUFFIXES,
                    ['present', 'area fraction', 'focus count', 'asymmetry'])
}

def _asymmetry(mask_u8):
    """1 - IoU(mask, mask mirrored about its own vertical centroid axis), in [0, 1].
    0 = perfectly mirror-symmetric about its own axis, 1 = no overlap at all.
    Rationale: dermoscopic pattern analysis and the ABCD rule both score the
    ASYMMETRY of structure distribution, not only whether a structure is there.
    Computed about the structure's own centroid so it measures the shape of the
    structure's distribution rather than where the lesion sits in the frame."""
    ys, xs = np.nonzero(mask_u8)
    if len(xs) == 0:
        return 0.0
    cx = xs.mean()
    """mirror x about cx, on a padded canvas so nothing falls off the edge"""
    pad = W
    canvas = np.zeros((H, 2 * pad + W), dtype=np.uint8)
    canvas[ys, xs + pad] = 1
    xm = np.rint(2 * (cx + pad) - (xs + pad)).astype(int)
    keep = (xm >= 0) & (xm < canvas.shape[1])
    mirror = np.zeros_like(canvas)
    mirror[ys[keep], xm[keep]] = 1
    inter = float((canvas & mirror).sum())
    union = float((canvas | mirror).sum())
    return 1.0 - (inter / union if union > 0 else 1.0)

def concepts_from_masks(masks, lesion_mask=None):
    """masks : (256, 256, 5) binary, ATTRIBUTES order  ->  (20,) float32.

    Per attribute, in CONCEPT_SUFFIXES order:
      present    1.0 if the attribute occupies at least PRESENCE_AREA_FRAC of the
                 reference area. This is the dimension the 7-point checklist style
                 of reasoning actually uses - criteria are scored present/absent.
      area_frac  positive area / reference area. Extent, not just presence: a
                 pigment network over the whole lesion reads differently from a
                 patch of it in one corner.
      n_blobs    connected components above MIN_BLOB_PX. Multifocality. Globules
                 and milia-like cysts are inherently multi-focus structures, and
                 'few large' vs 'many scattered' is a distinction clinicians draw.
      asymmetry  see _asymmetry. Irregular / asymmetric distribution of a structure
                 is what separates a *typical* from an *atypical* reading of it.

    lesion_mask : optional (256, 256) binary from the task-1 model. When given, the
    reference area is the lesion rather than the whole frame, which makes area_frac
    comparable across images shot at different magnifications. When None the frame
    is used and that comparability is lost - stated because it changes the meaning
    of the number, not just its scale."""
    if lesion_mask is not None and lesion_mask.sum() > 0:
        ref_area = float(lesion_mask.sum())
    else:
        ref_area = float(H * W)

    feats = []
    for c in range(N_ATTR):
        m = (masks[..., c] > 0.5).astype(np.uint8)
        if lesion_mask is not None and lesion_mask.sum() > 0:
            m = (m & (lesion_mask > 0.5).astype(np.uint8))

        area = float(m.sum())
        area_frac = area / ref_area

        if area == 0:
            feats.extend([0.0, 0.0, 0.0, 0.0])
            continue

        n_lab, lab = cv2.connectedComponents(m, connectivity=8)
        sizes = np.bincount(lab.ravel())
        sizes[0] = 0                                    # drop background
        n_blobs = float((sizes >= MIN_BLOB_PX).sum())

        present = 1.0 if area_frac >= PRESENCE_AREA_FRAC else 0.0
        asym = _asymmetry(m)

        feats.extend([present, area_frac, n_blobs, asym])

    return np.array(feats, dtype=np.float32)

print('bottleneck width:', N_CONCEPT, 'dimensions')
for k, name in enumerate(CONCEPT_NAMES):
    print(f"  [{k:2d}] {name:<34} {CONCEPT_PRETTY[name]}")

In [ ]:
def print_concept_vector(vec, title='concept vector'):
    """The bottleneck, printed with its names. This is the ENTIRE state that reaches
    the diagnosis. If a number here is wrong, the rationale built on it is wrong,
    however fluent it reads."""
    print(title)
    print('-' * len(title))
    for k, name in enumerate(CONCEPT_NAMES):
        print(f"  {name:<34} = {vec[k]:.6g}")

"""smoke test on ONE image that has attribute ground truth. Nothing is asserted
about the values - this only checks that the shapes line up and that the names
line up with the numbers, which is the bug that would silently poison every
rationale in the notebook."""
if attr_gt:
    demo_id = sorted(attr_gt)[0]
    demo_masks = read_attribute_masks(attr_gt[demo_id])
    demo_vec = concepts_from_masks(demo_masks)
    print('image id :', demo_id)
    print('masks    :', demo_masks.shape, ' vector:', demo_vec.shape,
          f'(expected ({N_CONCEPT},))')
    assert demo_vec.shape == (N_CONCEPT,)
    print()
    print_concept_vector(demo_vec, f'GROUND-TRUTH concept vector for {demo_id}')
else:
    print('no attribute ground truth resolved - cannot run the smoke test.')

# **Loading `g` and its fitted thresholds**

`g` is loaded with `compile=False` - the custom loss and metrics are irrelevant for inference and loading them would need task-2's measured `POS_WEIGHTS` in scope. The per-attribute thresholds are **loaded, never re-fitted here**: task 2 fits them on its own validation split, and re-fitting them against task-3 data would leak the label side into the concept side and quietly convert this from a sequential bottleneck into a jointly-trained one.

In [ ]:
"""g() : the pretrained task-2 attribute model. compile=False because the custom
loss and metrics are irrelevant for inference and loading them would require
task-2's measured POS_WEIGHTS to be in scope."""
def load_task2_model(path):
    if path is None or not os.path.exists(path):
        raise FileNotFoundError(
            'No trained task-2 model found. This notebook cannot invent one: the '
            'whole pipeline is g() then f(), and g() IS the task-2 U-Net. Train '
            'task2-attribute-detection.ipynb first, or attach its u_net_task2.h5.')
    return tf.keras.models.load_model(path, compile=False)

"""Per-attribute decision thresholds. Task 2 fits these on ITS OWN validation split
and saves them; 0.5 is the wrong operating point for a positively-weighted sigmoid.
They are LOADED, never re-fitted here - re-fitting them against task-3 data would
leak the label side into the concept side and quietly turn the bottleneck into a
jointly-trained one."""
thresholds_path = find_file(
    ['/kaggle/input/**/task2_best_thresholds.npy',
     '/kaggle/working/**/task2_best_thresholds.npy',
     '/content/drive/MyDrive/ISIC2018/Models/u_net_task2/task2_best_thresholds.npy'],
    'task-2 fitted thresholds')

if thresholds_path:
    ATTR_THRESHOLDS = np.load(thresholds_path).astype(np.float32)
    print('loaded fitted thresholds from task 2:',
          {a: round(float(t), 3) for a, t in zip(ATTRIBUTES, ATTR_THRESHOLDS)})
else:
    ATTR_THRESHOLDS = np.full(N_ATTR, 0.5, dtype=np.float32)
    print('NO saved thresholds found - falling back to 0.5 for every attribute.')
    print('Run task-2s threshold-search cell and np.save its `best_thresholds` as')
    print('  task2_best_thresholds.npy')
    print('before reading any number out of this notebook. At 0.5 the rare')
    print('attributes (streaks, negative network) will be systematically')
    print('under-detected, and every "absent" concept below inherits that bias -')
    print('which the rationale generator will then report as clinical absence.')

assert ATTR_THRESHOLDS.shape == (N_ATTR,)

In [ ]:
def predict_concepts(image_paths, model, thresholds=None, batch_size=8, lesion_model=None):
    """Run g() over a list of image paths -> (N, 20) predicted concept matrix.
    INFERENCE ONLY: the task-2 weights are never updated here, which is exactly
    what makes this the sequential regime."""
    thresholds = ATTR_THRESHOLDS if thresholds is None else thresholds
    C = np.zeros((len(image_paths), N_CONCEPT), dtype=np.float32)
    kept = []

    for start in tqdm(range(0, len(image_paths), batch_size),
                      total=math.ceil(len(image_paths) / batch_size)):
        chunk = image_paths[start:start + batch_size]
        batch, idxs = [], []
        for j, p in enumerate(chunk):
            x = read_image(p)
            if x is None:
                continue
            batch.append(x); idxs.append(start + j)
        if not batch:
            continue

        prob = model.predict(np.stack(batch), verbose=0)               ## (b, 256, 256, 5)
        pred = (prob >= thresholds.reshape(1, 1, 1, N_ATTR)).astype(np.float32)

        for b, i in enumerate(idxs):
            lesion = None
            if lesion_model is not None:
                lp = lesion_model.predict(batch[b][None, ...], verbose=0)[0, ..., 0]
                lesion = (lp >= 0.5).astype(np.float32)
            C[i] = concepts_from_masks(pred[b], lesion_mask=lesion)
            kept.append(i)

    return C, np.array(sorted(kept), dtype=int)

def gt_concepts(ids, id_to_paths):
    """(N, 20) concept matrix from the task-2 GROUND-TRUTH masks. This is c_true,
    what the independent and joint regimes need and what Route B has not got."""
    C = np.zeros((len(ids), N_CONCEPT), dtype=np.float32)
    for i, image_id in enumerate(tqdm(ids, total=len(ids))):
        C[i] = concepts_from_masks(read_attribute_masks(id_to_paths[image_id]))
    return C

# **Building the concept matrix**

One forward pass of `g` per image, then 20 numbers out. This is the slow cell and its output is cached to disk, because every regime, every sensitivity sweep and every rationale below reads the same matrix.

In [ ]:
"""====================================================================
   Build the concept matrix for whichever route was selected.
   No score is printed here; this only assembles matrices and shapes.
   ==================================================================="""
g_model = load_task2_model(task2_model_path)

if JOIN_MODE == 'gt_concepts':
    """ROUTE A - the intersection carries c_true and y together"""
    ids_all = list(JOIN_IDS)
    join_img_paths = [t12_images[i] for i in ids_all]
    C_true = gt_concepts(ids_all, attr_gt)
    C_pred, kept = predict_concepts(join_img_paths, g_model)
    ids_all = [ids_all[i] for i in kept]; C_pred = C_pred[kept]; C_true = C_true[kept]
    y_all = np.array([t3_labels[i] for i in ids_all], dtype=int)
    print('C_true', C_true.shape, ' C_pred', C_pred.shape, ' y', y_all.shape)

else:
    """ROUTE B - predicted concepts over the task-3 set. c_true does not exist for
    these images, by construction, and is left as None rather than as zeros: a
    zero matrix would silently flow into the regimes that must not run."""
    if not t3_images:
        raise RuntimeError(
            'No task-3 image directory resolved. The label predictor needs the task-3 '
            'IMAGES (not just the csv) to compute predicted concepts. Attach '
            'ISIC2018_Task3_Training_Input (or the HAM10000 image folders) and re-run '
            'the environment cell.')

    fallback_ids = sorted(set(t3_images) & set(t3_labels))
    print('task-3 ids with both a label and an image file:', len(fallback_ids))
    missing_imgs = len(set(t3_labels) - set(t3_images))
    if missing_imgs:
        print('  labelled ids with no image on disk:', missing_imgs)

    """optional stratified subsample - a U-Net forward pass per image over 10k
    images is the slow step. Set to None to use everything."""
    FALLBACK_N = 4000
    if FALLBACK_N is not None and FALLBACK_N < len(fallback_ids):
        y_full = np.array([t3_labels[i] for i in fallback_ids])
        keep, _ = train_test_split(np.arange(len(fallback_ids)), train_size=FALLBACK_N,
                                   stratify=y_full, random_state=42)
        fallback_ids = [fallback_ids[i] for i in sorted(keep)]
        print('subsampled (stratified) to', len(fallback_ids))

    C_pred, kept = predict_concepts([t3_images[i] for i in fallback_ids], g_model)
    ids_all = [fallback_ids[i] for i in kept]
    C_pred = C_pred[kept]
    y_all = np.array([t3_labels[i] for i in ids_all], dtype=int)
    C_true = None                       # there is no attribute GT here. None, not zeros.
    print('C_pred', C_pred.shape, ' y', y_all.shape, ' C_true =', C_true)

np.save(os.path.join(save_dir, 'C_pred.npy'), C_pred)
np.save(os.path.join(save_dir, 'y_all.npy'), y_all)
pd.DataFrame({'image': ids_all, 'label': y_all}).to_csv(
    os.path.join(save_dir, 'ids_and_labels.csv'), index=False)

"""one stratified split, reused by every regime, so the comparison is like-for-like"""
strat = y_all if np.bincount(y_all, minlength=N_CLASS).min() >= 2 else None
if strat is None:
    print('a class has <2 members - falling back to an unstratified split.')
idx_tr, idx_te = train_test_split(np.arange(len(y_all)), test_size=0.25,
                                  random_state=42, stratify=strat)
print('train', len(idx_tr), ' test', len(idx_te))
print('test-split class counts:', dict(zip(DIAGNOSES, np.bincount(y_all[idx_te], minlength=N_CLASS))))

# **The label predictor, and why it must be boring**

`f` is a **multinomial logistic regression** (and a depth-4 decision tree alongside it, for a
second reading). Not an MLP. This is the most consequential design decision in the notebook and
it is not about overfitting.

A concept bottleneck buys interpretability in two independent places, and they are easy to
conflate:

1. **The bottleneck is interpretable** - you can read off *what the model thinks it sees*. This
   holds for any `f`, even a 12-layer transformer on top.
2. **The reasoning is interpretable** - you can read off *how what it saw produced the answer*.
   This holds only if `f` itself is legible.

Put an MLP on top and you keep (1) and throw away (2). You would then be back to explaining `f`
post-hoc - SHAP values over 20 features, attributions of an attribution - which is the exact
regress a concept bottleneck was supposed to escape. Worse, the escape would be *invisible*: the
notebook would still print concept names and confident-looking contributions, and they would no
longer be the model.

With a linear `f`, the explanation is not an approximation of the model. It **is** the model:

```
logit(class j) = intercept[j] + sum_k coef[j, k] * z_k
```

where `z` is the standardised concept vector. `coef[j, k] * z_k` is the signed contribution of
concept `k` to class `j`, and those contributions sum to the logit *exactly*. The rationale
function below asserts that identity against sklearn's own `decision_function` and refuses to
print an explanation if it fails, because an explanation that does not reconstruct the prediction
is decoration.

Three practical notes:

* **`StandardScaler` is not cosmetic.** It puts all 20 dimensions on one scale so that
  `coef * value` products are comparable across dimensions. Without it, `area_frac` (order 1e-3)
  and `n_blobs` (order 10) produce contributions that cannot be ranked against each other.
* **`class_weight='balanced'`.** NV dominates HAM10000 by an order of magnitude. Unbalanced, `f`
  learns the prior and the "explanation" becomes a long-winded way of saying "usually a nevus".
  Task 3 attacked the same problem by resampling images; here it is one keyword.
* **The decision tree is there for a different kind of reading.** It is left *unscaled* on
  purpose: a split of the form `globules__area_frac <= <some fraction of the frame>` is a sentence
  a clinician can go and check against an image, and a split on a z-score is not. (The actual
  thresholds appear only when the cell runs; none is quoted here.) Depth 4 so the whole tree
  prints on one screen.
  Where the two models disagree is itself informative - a tree gets one decision path, a linear
  model weighs all 20 dimensions at once, and dermoscopy checklists are closer to the tree.


In [ ]:
"""====================================================================
   f(.) : concepts -> diagnosis. Deliberately shallow.
   ==================================================================="""
def make_logreg():
    """StandardScaler matters for more than convergence: it puts every concept on
    the same scale so that coef * standardised_value is directly comparable across
    dimensions. The rationale cell reads those products as contributions, which is
    only meaningful once the inputs share a scale.
    class_weight='balanced' because NV dominates - without it the model learns the
    prior, not the concepts."""
    return Pipeline([
        ('scale', StandardScaler()),
        ('clf', LogisticRegression(max_iter=5000, C=1.0, class_weight='balanced',
                                   random_state=42)),
    ])

def make_tree(max_depth=4):
    """depth 4 so the whole tree fits on a page and every path is readable.
    Unscaled on purpose: a threshold like 'globules__area_frac <= 0.031' is a
    sentence a clinician can check, a threshold on a z-score is not."""
    return DecisionTreeClassifier(max_depth=max_depth, class_weight='balanced',
                                  random_state=42)

def fit_eval(model, C_fit, y_fit, C_eval, y_eval, name=''):
    """returns the fitted model plus a small dict. No numbers are printed by this
    notebook's author - they appear only when the cell is run."""
    model.fit(C_fit, y_fit)
    pred = model.predict(C_eval)
    out = {
        'name': name,
        'accuracy': accuracy_score(y_eval, pred),
        'balanced_accuracy': balanced_accuracy_score(y_eval, pred),
        'macro_f1': f1_score(y_eval, pred, average='macro', zero_division=0),
    }
    return model, out, pred

def coef_frame(fitted_pipeline):
    """(7, 20) coefficient table - the whole model, in one printable object.
    THIS is the explanation. There is nothing else inside f(.)."""
    clf = fitted_pipeline.named_steps['clf']
    coefs = clf.coef_
    if coefs.shape[0] == 1:            # sklearn collapses a 2-class problem to one row
        coefs = np.vstack([-coefs[0], coefs[0]])
    present = [DIAGNOSES[k] for k in clf.classes_] if coefs.shape[0] == len(clf.classes_) else DIAGNOSES
    return pd.DataFrame(coefs, index=present, columns=CONCEPT_NAMES)

# **The three regimes**

Implemented in the order Koh et al. present them. Only sequential runs on this data; the other
two are gated by the measured intersection, with the gate printing why.


## The audit bench

Before the regimes: the join section established that the 7-class pipeline has no ground-truth
concepts, which closes off the independent regime, the joint regime, real concept fidelity on the
classified images, and the intervention experiment. All four are the interesting part.

They can be recovered on a **narrower** dataset. The Task 1-2 images do have diagnoses - not in the
Task 3 csv, but in the ISIC Archive metadata - so they can carry attribute masks *and* a label at
once. The cost is the class set: those legacy images are melanoma / nevus / seborrheic keratosis
only, three classes, no BCC, AKIEC, DF or VASC. And `g` was trained on most of them.

So the bench is not a better version of the pipeline. It is a **test rig**: the place where the
architecture's claims can actually be checked, at the price of being 3-class and optimistic. Every
number it produces is labelled as such, and none of them belongs in the headline comparison.

The fetch is opt-in (`FETCH_ISIC_DIAGNOSES`) because it makes network calls, and everything
downstream degrades gracefully with a printed explanation when it is off.


In [ ]:
"""====================================================================
   OPTIONAL: fetch diagnoses for the TASK 1-2 image ids from the ISIC
   Archive API. This is what makes the audit bench possible, and it is
   the only route to a genuinely joined (concepts + label) set here.

   OPT-IN, because it makes network calls and Kaggle kernels are often
   offline. Set FETCH_ISIC_DIAGNOSES = True to use it.
   ==================================================================="""
FETCH_ISIC_DIAGNOSES = False

ISIC_API = 'https://api.isic-archive.com/api/v2/images/search/'
"""ISIC Archive collection ids for the challenge splits. These reproduce the
official 2018 partitions exactly (the collection sizes match the challenge
downloads: 2594 / 100 / 1000 for Task 1-2 train/val/test)."""
ISIC_COLLECTIONS = {'task12_train': 63, 'task12_val': 62, 'task12_test': 64}

"""The Task 1-2 images are legacy-archive images and carry only THREE diagnoses -
melanoma, nevus, seborrheic keratosis - i.e. the ISIC 2017 class set. There is no
BCC, AKIEC, DF or VASC anywhere in this set. That is a hard limit on the audit
bench and it is why the bench cannot replace the 7-class pipeline.

Mapping into this notebook's label space is one-to-one for melanoma and nevus.
Seborrheic keratosis is NOT one-to-one: HAM10000's BKL is broader - 'benign
keratosis-like lesions' covers solar lentigines and lichen-planus-like keratoses
as well as seborrheic keratoses. So SK -> BKL is a SUBSET mapping, not an
equality, and a model trained on it has seen only part of what BKL means."""
ISIC_DIAGNOSIS_MAP = {
    'melanoma':               'MEL',
    'nevus':                  'NV',
    'seborrheic keratosis':   'BKL',   # subset of BKL, see above
}

def _norm_diagnosis(s):
    return ' '.join(str(s).lower().replace('_', ' ').split())

def fetch_isic_diagnoses(collection_ids, page_limit=100):
    """{ISIC_0000000: 'MEL'|'NV'|'BKL'} from the archive metadata.

    Written defensively on purpose: the API has changed shape between versions
    (flat `diagnosis` vs a hierarchical `diagnosis_1..5`), so rather than trusting
    one key this walks the clinical metadata for anything diagnosis-shaped, and
    PRINTS every value it could not map instead of dropping it silently. An
    unmapped diagnosis is a data-quality finding, not noise to be swallowed."""
    import json, urllib.request, urllib.parse

    out, unmapped, raw_seen = {}, {}, set()
    for name, cid in collection_ids.items():
        url = f"{ISIC_API}?{urllib.parse.urlencode({'collections': cid, 'limit': page_limit})}"
        n_this = 0
        while url:
            with urllib.request.urlopen(url, timeout=60) as r:
                payload = json.load(r)
            for item in payload.get('results', []):
                image_id = item.get('isic_id')
                meta = item.get('metadata', {}) or {}
                clinical = meta.get('clinical', {}) or {}
                """collect every diagnosis-ish value, most specific last"""
                cands = [v for k, v in sorted(clinical.items())
                         if 'diagnosis' in k.lower() and v]
                mapped = None
                for v in reversed(cands):
                    nv = _norm_diagnosis(v)
                    raw_seen.add(nv)
                    if nv in ISIC_DIAGNOSIS_MAP:
                        mapped = ISIC_DIAGNOSIS_MAP[nv]
                        break
                if mapped and image_id:
                    out[image_id] = mapped
                    n_this += 1
                elif image_id:
                    unmapped[image_id] = cands
            url = payload.get('next')
        print(f'  {name} (collection {cid}): {n_this} mapped')

    if unmapped:
        print(f'\n  {len(unmapped)} images with NO mappable diagnosis. Examples:')
        for k in list(unmapped)[:5]:
            print('   ', k, unmapped[k])
    print('\n  distinct raw diagnosis strings seen:', sorted(raw_seen))
    return out

t12_labels = {}
if FETCH_ISIC_DIAGNOSES:
    try:
        t12_labels = fetch_isic_diagnoses(ISIC_COLLECTIONS)
        print('\ntask 1-2 ids with a diagnosis:', len(t12_labels))
        vc = pd.Series(list(t12_labels.values())).value_counts()
        print(vc.to_string())
        print('\nExpected shape of this, from the published description of the split')
        print('(Mirikharaji et al., https://arxiv.org/abs/2206.00356): the 2594')
        print('training images are ~20% melanoma, ~72% nevi, ~8% seborrheic keratosis.')
        print('If the counts above are wildly different, something is wrong with the')
        print('collection ids or the metadata shape - check before using the bench.')
    except Exception as e:
        print('ISIC API fetch failed:', type(e).__name__, e)
        print('The audit bench will be skipped; the 7-class Route B pipeline is')
        print('unaffected. Kaggle kernels need "Internet" enabled for this cell.')
        t12_labels = {}
else:
    print('FETCH_ISIC_DIAGNOSES is False - skipping the network call.')
    print()
    print('What you are giving up by leaving it off, concretely:')
    print('  - the INDEPENDENT and JOINT regimes (both need c_true paired with y)')
    print('  - concept fidelity measured on the images actually being classified')
    print('  - the real Koh et al. concept-intervention experiment')
    print('Set it to True to build the audit bench. It is 3 classes, not 7, so it')
    print('does not replace the main pipeline - it audits it.')

In [ ]:
"""====================================================================
   THE AUDIT BENCH (Route A-prime).
   Task 1-2 images: attribute masks AND (via the API) a diagnosis.
   3 classes only - MEL / NV / BKL-as-seborrheic-keratosis.
   This is a smaller, narrower, HONESTLY JOINED dataset whose only job
   is to answer the questions Route B cannot.
   ==================================================================="""
AUDIT_OK = False
C_true_A = C_pred_A = y_A = None
ids_A = []
DIAG_A = ['MEL', 'NV', 'BKL']          # the only three present in this set

audit_ids = sorted(set(attr_gt) & set(t12_images) & set(t12_labels))
print('audit-bench candidates (attribute GT + image + diagnosis):', len(audit_ids))

if len(audit_ids) >= MIN_JOIN_TOTAL:
    y_A_codes = [t12_labels[i] for i in audit_ids]
    counts_A = pd.Series(y_A_codes).value_counts()
    print()
    print(counts_A.to_string())

    usable = [c for c in DIAG_A if counts_A.get(c, 0) >= MIN_PER_CLASS]
    if len(usable) >= 2:
        DIAG_A = usable
        keep = [i for i, c in zip(audit_ids, y_A_codes) if c in DIAG_A]
        ids_A = keep
        y_A = np.array([DIAG_A.index(t12_labels[i]) for i in ids_A], dtype=int)

        print()
        print('building c_true from the attribute masks...')
        C_true_A = gt_concepts(ids_A, attr_gt)
        print('building c_pred by running g() on the same images...')
        C_pred_A, kept_A = predict_concepts([t12_images[i] for i in ids_A], g_model)
        ids_A = [ids_A[i] for i in kept_A]
        C_true_A, C_pred_A, y_A = C_true_A[kept_A], C_pred_A[kept_A], y_A[kept_A]

        idx_tr_A, idx_te_A = train_test_split(
            np.arange(len(y_A)), test_size=0.25, random_state=42,
            stratify=y_A if np.bincount(y_A).min() >= 2 else None)

        AUDIT_OK = True
        print()
        print('AUDIT BENCH READY')
        print('  classes  :', DIAG_A)
        print('  C_true_A :', C_true_A.shape, ' C_pred_A:', C_pred_A.shape)
        print('  train/test:', len(idx_tr_A), '/', len(idx_te_A))
        print()
        print('IMPORTANT: g() WAS TRAINED ON MOST OF THESE IMAGES. The task-2 U-Net')
        print('saw the Task 1-2 training split, so concept fidelity and intervention')
        print('measured here are OPTIMISTIC - they show g() at its best, on its own')
        print('training distribution. Read them as an upper bound on how well the')
        print('concept extractor can behave, never as an estimate of Route B.')
    else:
        print('fewer than 2 classes clear MIN_PER_CLASS - audit bench not built.')
else:
    print('audit bench not available (need FETCH_ISIC_DIAGNOSES = True, network')
    print('access, and the task 1-2 images on disk).')
    print('Everything below that depends on it will say so and skip.')

### Regime 1 - independent

In [ ]:
"""====================================================================
   REGIME 1 - INDEPENDENT
   f fitted on TRUE concepts, g fitted on masks, neither aware the other
   exists. At test time f is fed g's PREDICTIONS.
   Koh et al. name the consequence exactly: "while f is trained using the
   true c, at test time it still takes g(x) as input". That mismatch
   costs plain accuracy and BUYS intervention response - corrected
   concepts are precisely the distribution f saw in training.
   Needs c_true + y on the same image -> AUDIT BENCH only, 3 classes.
   ==================================================================="""
results = {}

if AUDIT_OK:
    f_independent, r_ind, _ = fit_eval(
        make_logreg(),
        C_true_A[idx_tr_A], y_A[idx_tr_A],     # trained on GROUND-TRUTH concepts
        C_pred_A[idx_te_A], y_A[idx_te_A],     # tested on PREDICTED concepts
        name=f'BENCH independent, {len(DIAG_A)} classes (logreg)')
    results['independent'] = r_ind
    print(r_ind)

    """the train/test mismatch IS this regime, so measure it: same recipe, evaluated
    on true concepts. The gap is the damage done by g's errors with f held fixed.
    Upper bound, not a deployable system - at inference nobody has GT concepts."""
    _, r_ind_oracle, _ = fit_eval(
        make_logreg(),
        C_true_A[idx_tr_A], y_A[idx_tr_A],
        C_true_A[idx_te_A], y_A[idx_te_A],
        name=f'BENCH independent, ORACLE concepts at test time ({len(DIAG_A)} classes)')
    results['independent_oracle'] = r_ind_oracle
    print(r_ind_oracle)
    print()
    print('Both rows are 3-class and both are on images g() TRAINED ON. They are not')
    print('comparable to the 7-class sequential rows below and must never be quoted')
    print('as this pipeline accuracy.')
else:
    f_independent = None
    print('SKIPPED - independent training needs c_true paired with y on the same')
    print('image. Set FETCH_ISIC_DIAGNOSES = True to build the audit bench; without')
    print('it there is no such image anywhere in this repo.')

### Regime 2 - sequential

`g` is already trained and is frozen; `f` is fitted on `g`'s own outputs. This is the regime this
repo is shaped for, the only one available on the 7-class task, and - because at inference nobody
has attribute annotations - the one that matches deployment.

Its specific weakness is worth naming rather than glossing: because `f` is fitted on predicted
concepts, its coefficients are partly a statement about **`g`'s error pattern**, not only about the
concepts. If `g` never fires on streaks, `f` learns to ignore streaks, and a reader of the
coefficient table could mistake that for a clinical finding about streaks. Distinguishing those two
readings is exactly what the fidelity table is for, which is why it is not optional.

**This is the row to quote.** 7 classes, the same class set as the 0.615 black-box baseline.


In [ ]:
"""====================================================================
   REGIME 2 - SEQUENTIAL
   g is trained first and then FROZEN; f is trained on g's own outputs.
   Works on both routes - it never needs c_true - and is the natural fit here,
   because task 2 already produced a trained, frozen g.
   ==================================================================="""
f_sequential, r_seq, pred_seq = fit_eval(
    make_logreg(),
    C_pred[idx_tr], y_all[idx_tr],           # trained on PREDICTED concepts
    C_pred[idx_te], y_all[idx_te],
    name='sequential (logreg)')
results['sequential'] = r_seq
print(r_seq)

"""same regime, decision tree instead - the tree IS its own explanation"""
f_tree, r_tree, _ = fit_eval(
    make_tree(max_depth=4),
    C_pred[idx_tr], y_all[idx_tr],
    C_pred[idx_te], y_all[idx_te],
    name='sequential (decision tree, depth 4)')
results['sequential_tree'] = r_tree
print(r_tree)

print('\n--- the entire decision tree, as text ---')
print(export_text(f_tree, feature_names=CONCEPT_NAMES, decimals=4))

In [ ]:
"""per-class behaviour of the sequential model. Accuracy alone hides the thing
that matters clinically: MEL recall."""
print(classification_report(y_all[idx_te], pred_seq,
                            labels=list(range(N_CLASS)),
                            target_names=DIAGNOSES, zero_division=0))

cm = confusion_matrix(y_all[idx_te], pred_seq, labels=list(range(N_CLASS)))
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(cm, cmap='Blues')
ax.set_xticks(range(N_CLASS)); ax.set_xticklabels(DIAGNOSES, rotation=45)
ax.set_yticks(range(N_CLASS)); ax.set_yticklabels(DIAGNOSES)
ax.set_xlabel('predicted'); ax.set_ylabel('true')
ax.set_title('Sequential CBM - confusion matrix')
for i in range(N_CLASS):
    for j in range(N_CLASS):
        ax.text(j, i, cm[i, j], ha='center', va='center', fontsize=8)
fig.colorbar(im); plt.tight_layout(); plt.show()

"""coefficient heatmap - 7 x 20, the complete reasoning of f(.)"""
coefs = coef_frame(f_sequential)
fig, ax = plt.subplots(figsize=(16, 4))
lim = np.abs(coefs.to_numpy()).max()
im = ax.imshow(coefs.to_numpy(), cmap='RdBu_r', vmin=-lim, vmax=lim, aspect='auto')
ax.set_yticks(range(coefs.shape[0])); ax.set_yticklabels(coefs.index)
ax.set_xticks(range(N_CONCEPT)); ax.set_xticklabels(CONCEPT_NAMES, rotation=90, fontsize=7)
ax.set_title('Logistic-regression coefficients: red = pushes towards the class, blue = away')
fig.colorbar(im); plt.tight_layout(); plt.show()
coefs

### Regime 3 - joint

One objective, `L = L_label + lambda * L_concept`, so the concept extractor is allowed to move in
service of the diagnosis. Highest accuracy of the three in Koh et al.'s CUB experiment, and the most
compromised: the refinement that buys the accuracy is `g`'s outputs drifting away from meaning what
they are named, which is the leakage Margeloiu et al. and Mahinpei et al. document.

Bench only - it needs a mask target and a diagnosis in the same gradient step. And there is a
second, structural reason its accuracy is not comparable to anything else here. The differentiable
bottleneck can only be the spatial mean of each sigmoid map (a soft area fraction); `present`,
`n_blobs` and `asymmetry` all require thresholding and connected components, which have no usable
gradient. So the joint model's bottleneck is **5-dimensional, not 20** - narrower, and made of
exactly the continuous, most-leak-prone quantity of the four. Saying so is cheaper than pretending
the architectures match.

The `lambda` sweep at the end is the one experiment here that makes the accuracy-vs-interpretability
tradeoff visible as a curve rather than an assertion. Watch the concept dice alongside the accuracy:
if accuracy climbs while dice falls, that is leakage happening in front of you.


In [ ]:
"""====================================================================
   REGIME 3 - JOINT
   One network, two heads, one objective:
       L = L_label(f(g(x)), y)  +  LAMBDA * L_concept(g(x), c)
   LAMBDA -> 0   is the standard black box (a net with a narrow layer and
                 no reason for that layer to mean anything).
   LAMBDA -> inf is the sequential model.
   Needs mask GT and a diagnosis on the SAME image -> audit bench only.
   ==================================================================="""
LAMBDA = 1.0    # the accuracy/interpretability dial, not a constant. Sweep it.

def build_joint_cbm(task2_weights_path, n_class, trainable_g=True):
    """Reuse task-2's U-Net as g and bolt a linear label head onto a DIFFERENTIABLE
    bottleneck.

    Awkward but important: the differentiable bottleneck can only be the per-attribute
    spatial mean of the sigmoid map, i.e. a soft area fraction. The other three
    features per attribute (present / n_blobs / asymmetry) all involve thresholding
    and connected components, which have no usable gradient. So the JOINT model's
    bottleneck is 5-dimensional, not 20 - genuinely narrower than the sequential one -
    and its accuracy is NOT directly comparable to any other row here. Saying so is
    cheaper than pretending the architectures match.

    Note also that a soft area fraction is exactly the kind of continuous concept
    score Mahinpei et al. show leaks most, so this regime is simultaneously the least
    comparable and the most exposed to concept leakage."""
    g = load_task2_model(task2_weights_path)
    g.trainable = trainable_g

    mask_out = g.output                                                     ## (None,256,256,5)
    concept  = GlobalAveragePooling2D(name='concept_bottleneck')(mask_out)  ## (None, 5)

    """No hidden layer. Dense straight off the bottleneck keeps f linear and therefore
    readable - see the markdown on why an MLP here defeats the point."""
    logits = Dense(n_class, activation='softmax', name='diagnosis')(concept)
    return Model(g.input, [mask_out, logits], name='joint_CBM'), g

if AUDIT_OK:
    joint_model, g_backbone = build_joint_cbm(task2_model_path, n_class=len(DIAG_A))

    """outputs given as a LIST in build_joint_cbm's return order: [mask_out, logits].
    LAMBDA weights the CONCEPT term, which is where Koh et al. put it, so LAMBDA = 0
    really does reduce this to an unsupervised 5-unit bottleneck."""
    joint_model.compile(optimizer=Adam(1e-4),
                        loss=[dice_loss_multilabel, 'categorical_crossentropy'],
                        loss_weights=[LAMBDA, 1.0],
                        metrics=[[dice_macro], ['accuracy']])
    joint_model.summary()
    print('\nbottleneck output shape:',
          joint_model.get_layer('concept_bottleneck').output_shape,
          ' <- (None, 5) soft area fractions, NOT the 20-dim vector')
    print('loss = 1.0 * CE(y) + LAMBDA * dice_loss_multilabel(c), LAMBDA =', LAMBDA)
    print('classes:', DIAG_A)
else:
    joint_model = None
    print('SKIPPED - joint training needs a mask target and a diagnosis in the same')
    print('gradient step. Needs the audit bench.')

In [ ]:
def joint_generator(ids, id_to_paths, images, labels, diag, batch_size=4, shuffle_each_epoch=True):
    """yields (image_batch, [mask_batch, onehot_batch]).
    The joint regime needs the concept target and the label target for the SAME image
    in the SAME gradient step - exactly the pairing Route B cannot supply."""
    order = list(ids)
    while True:
        if shuffle_each_epoch:
            random.shuffle(order)
        for start in range(0, len(order) - batch_size + 1, batch_size):
            xb, mb, yb = [], [], []
            for image_id in order[start:start + batch_size]:
                x = read_image(images[image_id])
                if x is None:
                    continue
                xb.append(x)
                mb.append(read_attribute_masks(id_to_paths[image_id]))
                oh = np.zeros(len(diag), dtype=np.float32)
                oh[diag.index(labels[image_id])] = 1.0
                yb.append(oh)
            if xb:
                yield np.stack(xb), [np.stack(mb), np.stack(yb)]

if joint_model is not None:
    joint_batch, joint_epochs = 4, 20
    tr_ids = [ids_A[i] for i in idx_tr_A]
    te_ids = [ids_A[i] for i in idx_te_A]

    joint_history = joint_model.fit(
        joint_generator(tr_ids, attr_gt, t12_images, t12_labels, DIAG_A, joint_batch),
        steps_per_epoch=max(1, len(tr_ids) // joint_batch),
        validation_data=joint_generator(te_ids, attr_gt, t12_images, t12_labels,
                                        DIAG_A, joint_batch, False),
        validation_steps=max(1, len(te_ids) // joint_batch),
        epochs=joint_epochs,
        callbacks=[
            ModelCheckpoint(f"{save_dir}/joint_cbm.h5", monitor='val_loss',
                            verbose=1, save_best_only=True),
            ReduceLROnPlateau(monitor='val_loss', factor=0.1, patience=4,
                              min_lr=1e-7, verbose=1),
            CSVLogger(f"{save_dir}/joint_cbm.csv"),
            EarlyStopping(monitor='val_loss', patience=8, restore_best_weights=True),
        ])

    """LAMBDA sweep - the accuracy-vs-interpretability dial made literal. Re-run the
    two cells above per value and tabulate label accuracy against concept dice. Left
    as a grid rather than one value because a single LAMBDA says nothing about the
    SHAPE of the tradeoff, which is the only interesting part."""
    LAMBDA_GRID = [0.0, 0.01, 0.1, 1.0, 10.0]
    print('to sweep:', LAMBDA_GRID)
    print('record val diagnosis accuracy AND val dice_macro for each. Koh et al. chose')
    print('lambda by "highest task accuracy while maintaining high concept accuracy on')
    print('the validation set" - lambda=1 for OAI, 0.01 for CUB.')
    print()
    print('WATCH THE CONCEPT DICE, not just the accuracy. If accuracy climbs while')
    print('dice falls, that is leakage happening in front of you: the bottleneck is')
    print('becoming useful by ceasing to mean what its channels are named.')
else:
    joint_history = None
    print('no joint model to fit.')

# **Concept fidelity: is `g` actually right about the concepts?**

This is the cell that decides whether anything above is worth reading. A concept bottleneck whose
concepts are wrong is not a partially-interpretable model - it is a black box wearing a costume,
because the rationale still reads fluently and is simply fiction. Mahinpei et al.'s framing
applies directly: the usefulness of a concept representation is *not* evidence that the human
concept relates to the label.

Two different questions get answered separately:

* **Mask fidelity** - pooled per-attribute Jaccard of predicted against ground-truth pixels,
  using task 2's pooled-over-the-dataset definition rather than per-image averaging (most images
  have no positive pixels for most attributes, which makes a per-image Jaccard undefined).
* **Concept fidelity** - does the 20-dim *vector* agree? For `present` that is an AUC question,
  scored against `g`'s max sigmoid so it is threshold-free and does not merely re-measure the
  thresholds. For the continuous dimensions, a correlation.

Calibrate before reading: the ISIC 2018 Task 2 **winner** averaged **0.292** Jaccard - pigment
network 0.563 down to streaks 0.156 (<https://arxiv.org/abs/2104.01641>) - against a Task 1
leaderboard topping out near **0.80** (<https://challenge.isic-archive.com/leaderboards/2018/>).
The best in the world at attribute detection is about a third as good as a plain U-Net is at
lesion boundaries. A macro Jaccard of 0.1-0.2 from this repo's single un-pretrained U-Net is a
normal outcome, not a bug - and it is also a direct statement about how much the rationales below
should be trusted, attribute by attribute.

**And the measurement is on the wrong images.** Fidelity here is computed on held-out Task 2
images. `f` runs on Task 3 images. The two sets are disjoint acquisition batches. Assuming
fidelity transfers is an assumption, not a measurement, and it is the load-bearing one.


In [ ]:
"""====================================================================
   CONCEPT FIDELITY. A CBM whose concepts are wrong is a black box in a
   costume: the rationale still reads fluently and is still fiction.
   Measurable ONLY where attribute ground truth exists, i.e. on the
   task-2 images -- NEVER on the task-3 images f() is trained on.
   ==================================================================="""
FID_HOLDOUT = 0.15   # held-out share of the task-2 set, so g is not scored on its own train split

def task2_holdout_ids(attr_gt, images, holdout=FID_HOLDOUT, seed=42):
    """Reproduces task 2's own split protocol closely enough to be an honest
    held-out set: same random_state=42, same fraction-of-total logic. It is NOT
    guaranteed to be byte-identical to the split task 2 used, so a few of these
    images may have been seen by g during task-2 training. Flagged, not hidden -
    fidelity measured here is therefore an optimistic estimate."""
    usable = sorted(set(attr_gt) & set(images))
    if len(usable) < 10:
        return []
    _, hold = train_test_split(usable, test_size=holdout, random_state=seed)
    return sorted(hold)

FID_IDS = task2_holdout_ids(attr_gt, t12_images)
n_usable_fid = len(set(attr_gt) & set(t12_images))
print('task-2 ids usable for fidelity :', n_usable_fid)
print('held-out subset scored below   :', len(FID_IDS))

"""the official task-2 TRAINING split is 2594 images. If the resolved directory has
noticeably more, it is probably the train/val/test splits concatenated
(2594 + 100 + 1000 = 3694), which means a slice of it was in g's training data."""
OFFICIAL_T2_TRAIN = 2594
if n_usable_fid > OFFICIAL_T2_TRAIN:
    print()
    print('WARNING:', n_usable_fid, 'ids resolved, more than the official',
          OFFICIAL_T2_TRAIN, 'training images.')
    print('  That directory is probably the train/val/test splits concatenated.')
    print('  Consequence: the "held-out" subset below is drawn from a pool that')
    print('  includes the training images of g(), so the fidelity numbers are')
    print('  OPTIMISTIC by an amount this notebook cannot measure. To fix it, restrict')
    print('  attr_gt to the official training id list before splitting.')

def concept_fidelity(ids, id_to_paths, images, model, thresholds):
    """Two different questions, both worth asking separately:
      (a) MASK fidelity    - pooled per-attribute Jaccard of predicted vs GT pixels,
          using task 2's pooled-over-the-dataset definition, not per-image averaging.
      (b) CONCEPT fidelity - does the 20-dim vector agree? For the binary `present`
          dimension that is an AUC question (threshold-free, so it does not merely
          re-measure the thresholds); for the continuous ones, a correlation."""
    inter = np.zeros(N_ATTR); union = np.zeros(N_ATTR)
    pres_true, pres_score, cont_true, cont_pred = [], [], [], []

    for image_id in tqdm(ids, total=len(ids)):
        x = read_image(images[image_id])
        if x is None:
            continue
        y = read_attribute_masks(id_to_paths[image_id])
        prob = model.predict(x[None, ...], verbose=0)[0]
        pred = (prob >= thresholds.reshape(1, 1, N_ATTR)).astype(np.float32)

        yf, pf = y.reshape(-1, N_ATTR), pred.reshape(-1, N_ATTR)
        i_ch = (yf * pf).sum(0)
        inter += i_ch
        union += yf.sum(0) + pf.sum(0) - i_ch

        c_t = concepts_from_masks(y)
        c_p = concepts_from_masks(pred)
        pres_true.append([c_t[k * len(CONCEPT_SUFFIXES)] for k in range(N_ATTR)])
        pres_score.append([float(prob[..., k].max()) for k in range(N_ATTR)])
        cont_true.append(c_t); cont_pred.append(c_p)

    pres_true = np.array(pres_true); pres_score = np.array(pres_score)
    cont_true = np.array(cont_true); cont_pred = np.array(cont_pred)

    rows = []
    for k, a in enumerate(ATTRIBUTES):
        yt = pres_true[:, k]
        auc = roc_auc_score(yt, pres_score[:, k]) if 0 < yt.sum() < len(yt) else np.nan
        pk = cont_pred[:, k * len(CONCEPT_SUFFIXES)]
        rows.append({
            'Attribute': ATTR_LABELS[a],
            'mask Jaccard (pooled)': inter[k] / union[k] if union[k] > 0 else np.nan,
            'presence AUC': auc,
            'presence F1': f1_score(yt, pk, zero_division=0),
            'n with attribute (GT)': int(yt.sum()),
            'n predicted present': int(pk.sum()),
        })
    mask_df = pd.DataFrame(rows)

    dim_rows = []
    for k, name in enumerate(CONCEPT_NAMES):
        t, p = cont_true[:, k], cont_pred[:, k]
        r = np.nan if (t.std() < 1e-12 or p.std() < 1e-12) else float(np.corrcoef(t, p)[0, 1])
        dim_rows.append({'concept': name, 'pearson r (pred vs GT)': r,
                         'GT mean': t.mean(), 'pred mean': p.mean()})
    return mask_df, pd.DataFrame(dim_rows)

if len(FID_IDS) > 0:
    fid_mask_df, fid_dim_df = concept_fidelity(FID_IDS, attr_gt, t12_images,
                                               g_model, ATTR_THRESHOLDS)
    print()
    print(fid_mask_df.to_string(index=False))
    print()
    print(fid_dim_df.to_string(index=False))
    fid_mask_df.to_csv(os.path.join(save_dir, 'concept_fidelity_attributes.csv'), index=False)
    fid_dim_df.to_csv(os.path.join(save_dir, 'concept_fidelity_dimensions.csv'), index=False)
    print()
    print('READ THIS WITH THE TRANSFER CAVEAT. These scores are measured on task-2')
    print('images (ISIC_00xxxxx). The label predictor runs on task-3 / HAM10000')
    print('images (ISIC_002xxxx-003xxxx) - a different acquisition batch, different')
    print('centres, different cameras. Assuming this fidelity carries over is an')
    print('ASSUMPTION, and it is the single largest unverified step in the pipeline.')
else:
    fid_mask_df = fid_dim_df = None
    print('No task-2 images with both GT and image files - fidelity NOT MEASURABLE.')
    print('That is not a neutral gap: with no fidelity number, every rationale this')
    print('notebook prints is unaudited and could be fluent and wrong.')

In [ ]:
"""Fidelity on the AUDIT BENCH - i.e. on the very images the bench classifier is
scored on. This is the measurement Route B cannot make at all, and it is the one
that licenses reading the bench's rationales.

Two caveats, both large, both unavoidable:
  1. g() was TRAINED on most of these images, so this is g() at its best.
  2. 3 classes, legacy-archive images. Nothing here estimates Route B."""
if AUDIT_OK:
    bench_te_ids = [ids_A[i] for i in idx_te_A]
    bench_mask_df, bench_dim_df = concept_fidelity(bench_te_ids, attr_gt, t12_images,
                                                   g_model, ATTR_THRESHOLDS)
    print(bench_mask_df.to_string(index=False))
    print()
    print(bench_dim_df.to_string(index=False))
    bench_mask_df.to_csv(os.path.join(save_dir, 'bench_fidelity.csv'), index=False)
    print()
    print('Compare these row by row against the held-out task-2 table above. Where')
    print('the bench is much better, the difference is memorisation: g() saw these')
    print('images. Where both are poor, that attribute is simply beyond this g(), and')
    print('any rationale clause citing it is unsupported on BOTH datasets.')
else:
    bench_mask_df = bench_dim_df = None
    print('no audit bench - skipped.')

# **The payoff: image in, explanation out**

Everything above exists for this cell. `explain()` takes one image path and prints the predicted
diagnosis, the concepts `g` found, the signed contribution of each concept to the winning class, the
arithmetic showing those contributions reconstruct the model's logit exactly, and the clinical note
for each detected structure - alongside the attribute-mask overlay in task 2's colour scheme.

The assertion in the middle of the function is the important line. If the contributions ever fail to
sum to sklearn's own `decision_function` output, the function raises instead of printing: an
explanation that does not reproduce the prediction is decoration, and printing it anyway would be
the exact failure this whole notebook argues against.


### Output format

The next cell defines `explain()`. When run, it prints a block shaped like this - **this is a
layout schematic with the numbers deliberately left as `x`, not a result.** Nothing in this
notebook has been run and no rationale has ever been generated.

```
Predicted diagnosis : <CODE> (<name>), p = x.xxx
Runner-up           : <CODE>, p = x.xxx

Concepts detected in this image (at the task-2 fitted thresholds):
  - <Attribute>        area x.xx% of frame, x focus/foci, asymmetry x.xx

Why <CODE> - the 6 largest signed contributions to its score:
  + x.xxx   <Attribute> - <feature>            (value x.xxxx, coef +x.xxx)
  - x.xxx   <Attribute> - <feature>            (value x.xxxx, coef -x.xxx)
  ...

  logit(<CODE>) = intercept +x.xxx + sum of ALL 20 contributions = +x.xxx

Clinical reading of the concepts above (...):
  - <Attribute>: <the CLINICAL_NOTE text for that attribute>
```

plus the attribute-overlay figure: original image, predicted concepts composited in the task-2
colour scheme, attribute ground truth where it exists (it does not on Route B), and the five
per-channel masks.

Read the sign column, not the magnitude column, first: a large positive contribution to MEL from
a concept whose fidelity table row is poor is the notebook telling you not to believe it.


In [ ]:
"""====================================================================
   THE PAYOFF: image -> concepts -> diagnosis -> a sentence.
   ==================================================================="""
def overlay_masks(ori_bgr, masks, alpha=0.45, mode='fill'):
    """same overlay idiom as task 2 so the two notebooks' figures are readable
    against each other. masks : (256, 256, 5) in {0,1}."""
    out = ori_bgr.copy().astype(np.uint8)
    for c, a in enumerate(ATTRIBUTES):
        m = (masks[..., c] > 0.5).astype(np.uint8)
        if m.sum() == 0:
            continue
        color = ATTR_COLORS_BGR[a]
        if mode == 'fill':
            layer = out.copy()
            layer[m == 1] = color
            out = cv2.addWeighted(layer, alpha, out, 1 - alpha, 0)
        else:
            contours, _ = cv2.findContours(m, cv2.RETR_LIST, cv2.CHAIN_APPROX_SIMPLE)
            cv2.drawContours(out, contours, -1, color, 1)
    return out

def bgr2rgb(x):
    return cv2.cvtColor(x.astype(np.uint8), cv2.COLOR_BGR2RGB)

def explain(image_path, g=None, f=None, top_k=6, show=True, gt_mask_paths=None):
    """One image in, one explanation out.

    The signed contribution of concept k to class j is
        contribution[j, k] = coef[j, k] * z_k          (z = standardised concept)
    and the class logit is exactly intercept[j] + sum_k contribution[j, k].
    So these contributions are not a post-hoc attribution of a black box - they
    ARE the model. The sum is checked against the model's own decision_function
    below; if that assertion ever fails, the explanation has drifted from the
    prediction and must not be trusted."""
    g = g if g is not None else g_model
    f = f if f is not None else f_sequential

    x = read_image(image_path)
    if x is None:
        raise FileNotFoundError(image_path)
    ori = (x * 255).astype(np.uint8)

    prob_masks = g.predict(x[None, ...], verbose=0)[0]
    pred_masks = (prob_masks >= ATTR_THRESHOLDS.reshape(1, 1, N_ATTR)).astype(np.float32)
    c = concepts_from_masks(pred_masks)

    clf = f.named_steps['clf']
    proba = f.predict_proba(c[None, :])[0]
    order = np.argsort(-proba)
    top_class_pos = int(order[0])
    top_class = int(clf.classes_[top_class_pos])

    """sklearn stores one coefficient row per class for multinomial, but collapses a
    2-class problem to a single row. Expand it so the indexing below is uniform."""
    coefs, intercepts = clf.coef_, clf.intercept_
    if coefs.shape[0] == 1:
        coefs = np.vstack([-coefs[0], coefs[0]])
        intercepts = np.array([-intercepts[0], intercepts[0]])

    z = f.named_steps['scale'].transform(c[None, :])[0]
    row = coefs[top_class_pos]
    contrib = row * z

    """the explanation must reconstruct the model's own score, exactly"""
    logit = float(intercepts[top_class_pos]) + float(contrib.sum())
    df_check = np.atleast_2d(f.decision_function(c[None, :]))
    ref = float(df_check[0][top_class_pos]) if df_check.shape[1] > 1 else float(df_check[0][0])
    assert abs(logit - ref) < 1e-6, (
        'contributions do not sum to the model logit - do not trust this rationale')

    rank = np.argsort(-np.abs(contrib))[:top_k]

    lines = []
    lines.append(f"Predicted diagnosis : {DIAGNOSES[top_class]} "
                 f"({DIAGNOSIS_LABELS[DIAGNOSES[top_class]]}), p = {proba[top_class_pos]:.3f}")
    runner = order[1]
    lines.append(f"Runner-up           : {DIAGNOSES[int(clf.classes_[runner])]}, "
                 f"p = {proba[runner]:.3f}")
    lines.append("")
    lines.append("Concepts detected in this image (at the task-2 fitted thresholds):")
    any_present = False
    for k, a in enumerate(ATTRIBUTES):
        base = k * len(CONCEPT_SUFFIXES)
        if c[base] > 0.5:
            any_present = True
            lines.append(f"  - {ATTR_LABELS[a]:<18} area {c[base+1]*100:.2f}% of frame, "
                         f"{int(c[base+2])} focus/foci, asymmetry {c[base+3]:.2f}")
    if not any_present:
        lines.append("  - none of the five above threshold. The prediction below therefore")
        lines.append("    rests on ABSENCE of concepts, which is much weaker evidence, and")
        lines.append("    may equally mean the attribute model missed them.")
    lines.append("")
    lines.append(f"Why {DIAGNOSES[top_class]} - the {top_k} largest signed contributions to its score:")
    for k in rank:
        sign = '+' if contrib[k] > 0 else '-'
        lines.append(f"  {sign} {abs(contrib[k]):.3f}   {CONCEPT_PRETTY[CONCEPT_NAMES[k]]:<38}"
                     f" (value {c[k]:.4g}, coef {row[k]:+.3f})")
    lines.append("")
    lines.append(f"  logit({DIAGNOSES[top_class]}) = intercept {float(intercepts[top_class_pos]):+.3f}"
                 f" + sum of ALL {N_CONCEPT} contributions = {logit:+.3f}")
    lines.append("")
    lines.append("Clinical reading of the concepts above (see the clinical-grounding section;")
    lines.append("these are textbook associations, NOT this model's findings, and the")
    lines.append("mapping from a segmentation mask to a clinical descriptor has not been")
    lines.append("checked by a dermatologist):")
    for k, a in enumerate(ATTRIBUTES):
        if c[k * len(CONCEPT_SUFFIXES)] > 0.5:
            lines.append(f"  - {ATTR_LABELS[a]}: {CLINICAL_NOTE[a]}")

    text = '\n'.join(lines)
    if show:
        print(text)
        panels = 2 + (1 if gt_mask_paths is not None else 0)
        fig, ax = plt.subplots(1, panels + N_ATTR, figsize=(4 * (panels + N_ATTR) / 1.6, 3.2))
        ax[0].imshow(bgr2rgb(ori)); ax[0].set_title(os.path.basename(image_path), fontsize=8)
        ax[1].imshow(bgr2rgb(overlay_masks(ori, pred_masks, mode='fill')))
        ax[1].set_title('predicted concepts', fontsize=8)
        off = 2
        if gt_mask_paths is not None:
            ax[2].imshow(bgr2rgb(overlay_masks(ori, read_attribute_masks(gt_mask_paths), mode='fill')))
            ax[2].set_title('attribute GT', fontsize=8)
            off = 3
        for k, a in enumerate(ATTRIBUTES):
            ax[off + k].imshow(pred_masks[..., k], cmap='gray', vmin=0, vmax=1)
            ax[off + k].set_title(ATTR_LABELS[a], fontsize=7)
        for a_ in ax:
            a_.axis('off')
        handles = [mpatches.Patch(color=np.array(ATTR_COLORS_BGR[a][::-1]) / 255.0,
                                  label=ATTR_LABELS[a]) for a in ATTRIBUTES]
        fig.legend(handles=handles, loc='lower center', ncol=5, fontsize=7)
        plt.tight_layout(); plt.show()

    return {'concepts': c, 'proba': proba, 'contributions': contrib,
            'pred_class': DIAGNOSES[top_class], 'text': text, 'masks': pred_masks}

In [ ]:
"""Run it on a few test-split images. Whatever prints IS the notebook's output;
nothing above presupposes what it will say."""
n_show = 3
for i in idx_te[:n_show]:
    image_id = ids_all[i]
    if JOIN_MODE == 'gt_concepts':
        img_path, mask_paths = t12_images[image_id], attr_gt[image_id]
    else:
        img_path, mask_paths = t3_images[image_id], None   # no attribute GT exists here
    print('=' * 78)
    print('image             :', image_id)
    print('true diagnosis    :', DIAGNOSES[t3_labels[image_id]],
          f"({DIAGNOSIS_LABELS[DIAGNOSES[t3_labels[image_id]]]})")
    print('melanocytic class :', DIAGNOSES[t3_labels[image_id]] in MELANOCYTIC,
          '  <- if False, the 5 attributes were never meant to describe this lesion')
    print()
    explain(img_path, gt_mask_paths=mask_paths)
    print()

# **Concept intervention - the real experiment, and its stand-in**

This is the demonstration that most clearly justifies a concept bottleneck. Koh et al.'s version:
replace a predicted concept with its **ground-truth** value, re-run `f`, and measure whether accuracy
improves. On OAI, querying just two concepts took task RMSE from above 0.4 to about 0.3. The result
also discriminates between regimes - independent bottlenecks respond best, because corrected concepts
are the distribution `f` was trained on, while sequential and joint models can get *worse* under
intervention.

**On the 7-class pipeline this is impossible**, because no HAM10000 image has attribute ground truth.
It runs on the audit bench instead, where the concepts are real and the correction is real - with the
standing caveat that `g` was trained on those images, so the "before" accuracy is flattering and the
measured gap is a lower bound on how much concept error costs in the wild.

For the 7-class pipeline what is left is a **sensitivity analysis** further down: set a concept to a
chosen value and see how the prediction moves. It answers the strictly weaker question *how much does
`f` lean on this concept*, and it **cannot report an accuracy gain** because there is no true value to
move towards. Its function signature does not even accept the labels.


In [ ]:
"""====================================================================
   THE REAL INTERVENTION EXPERIMENT (Koh et al.).
   Overwrite a PREDICTED concept with its GROUND-TRUTH value, re-run
   f(), and measure whether accuracy improves. Unlike the synthetic
   sweep below, this HAS a correct answer to move towards, so it can
   and does report accuracy - and it splits every error into a concept
   error or a reasoning error.
   Audit bench only: it needs attribute GT on the classified images.
   ==================================================================="""
def intervene(C_pred_m, C_true_m, y, f, which='all', diag=None):
    """which : 'all' replaces the whole 20-dim vector with the GT-derived one;
    an attribute name replaces only that attribute's 4 dimensions."""
    diag = diag if diag is not None else DIAG_A
    C_fixed = C_true_m.copy() if which == 'all' else C_pred_m.copy()
    if which != 'all':
        k = ATTRIBUTES.index(which)
        sl = slice(k * len(CONCEPT_SUFFIXES), (k + 1) * len(CONCEPT_SUFFIXES))
        C_fixed[:, sl] = C_true_m[:, sl]

    before = f.predict(C_pred_m)
    after = f.predict(C_fixed)
    return pd.DataFrame({
        'true':   [diag[c] for c in y],
        'before': [diag[c] for c in before],
        'after':  [diag[c] for c in after],
        'changed':        before != after,
        'before correct': before == y,
        'after correct':  after == y,
    })

if AUDIT_OK:
    """f() for the bench: sequential regime, fitted on PREDICTED concepts, which is
    the regime whose response to intervention is least favourable - Koh et al. note
    interventions can even hurt sequential/joint models, because corrected concepts
    are off the distribution f() was fitted on. Reporting the harder case."""
    f_bench, r_bench, _ = fit_eval(
        make_logreg(),
        C_pred_A[idx_tr_A], y_A[idx_tr_A],
        C_pred_A[idx_te_A], y_A[idx_te_A],
        name=f'audit bench, sequential, {len(DIAG_A)} classes')
    print(r_bench)

    iv_all = intervene(C_pred_A[idx_te_A], C_true_A[idx_te_A], y_A[idx_te_A], f_bench)
    print()
    print('accuracy with PREDICTED concepts :', iv_all['before correct'].mean())
    print('accuracy with GT concepts        :', iv_all['after correct'].mean())
    print('predictions changed              :', int(iv_all['changed'].sum()),
          '/', len(iv_all))
    print()
    print('The gap between those two accuracies is the share of error attributable')
    print('to CONCEPT error rather than to f() reasoning. If correcting every')
    print('concept barely moves accuracy, the concepts were not the bottleneck and')
    print('f() is the weak link. If it jumps, g() is. A black box cannot be asked')
    print('this question at all - there is no named quantity to correct.')

    """per-attribute: which single corrected concept is worth the most?
    This is the actionable version - it says which channel of g() to fix first."""
    per_attr = [{'corrected attribute': ATTR_LABELS[a],
                 'accuracy after': intervene(C_pred_A[idx_te_A], C_true_A[idx_te_A],
                                             y_A[idx_te_A], f_bench,
                                             which=a)['after correct'].mean()}
                for a in ATTRIBUTES]
    per_attr_df = pd.DataFrame(per_attr)
    per_attr_df['gain over no correction'] = (per_attr_df['accuracy after']
                                              - iv_all['before correct'].mean())
    print()
    print(per_attr_df.sort_values('gain over no correction', ascending=False)
                     .to_string(index=False))
    per_attr_df.to_csv(os.path.join(save_dir, 'intervention_per_attribute.csv'),
                       index=False)
else:
    iv_all = None
    print('SKIPPED - the real intervention experiment needs the audit bench, which')
    print('needs attribute ground truth on the images being classified.')
    print('On Route B alone this experiment is impossible, not merely inconvenient.')

# **Failure attribution**

The question a concept bottleneck is uniquely able to answer: when the model is wrong, was it a
**concept error** (`g` misread the image, `f` reasoned correctly from wrong inputs) or a **reasoning
error** (`g` was right and `f` still blew it)? With ground-truth concepts it is settled per image -
correct the concepts, re-predict, see whether the error disappears.

The cell below also reports the case nobody looks for: images that were **right with predicted
concepts and wrong once corrected**. Those are predictions that were relying on a concept error to
land on the right answer - right for the wrong reason. A black box would have scored every one of
them as a success, and no amount of staring at a saliency map would have revealed it.

Bench only. The 7-class stand-in is the risk triage two cells down.


In [ ]:
"""====================================================================
   FAILURE ATTRIBUTION, the clean version - only on the audit bench.
   Every error goes into exactly one box:
     CONCEPT error   - correcting the concepts fixes the prediction, so
                       f() reasoned correctly from wrong inputs.
     REASONING error - the concepts were right and f() still failed.
   ==================================================================="""
if iv_all is not None:
    err = iv_all[~iv_all['before correct']].copy()
    err['attributed to'] = np.where(err['after correct'],
                                    'CONCEPT error', 'REASONING error')
    print('test images :', len(iv_all))
    print('errors      :', len(err))
    print()
    print(err['attributed to'].value_counts().to_string())
    print()
    """the reverse case is just as informative and usually ignored: images that were
    RIGHT with predicted concepts and WRONG once corrected. Those are cases where
    f() was relying on a concept error to get the right answer - right for the wrong
    reason, which a black box would have scored as a success."""
    lucky = iv_all[iv_all['before correct'] & ~iv_all['after correct']]
    print('right-for-the-wrong-reason (correct before, wrong after correction):',
          len(lucky))
    print('Those would all have counted as successes for a black box.')
    print()
    print(err.head(20).to_string(index=False))
    err.to_csv(os.path.join(save_dir, 'failure_attribution.csv'), index=False)
else:
    print('needs the audit-bench intervention results.')

### The 7-class stand-in: synthetic sensitivity

No ground truth, so no correction and no accuracy. Read it as a probe of `f`, not a test of `g`.

In [ ]:
"""====================================================================
   SYNTHETIC CONCEPT INTERVENTION (a SENSITIVITY ANALYSIS).

   *** THIS IS NOT THE KOH ET AL. INTERVENTION EXPERIMENT. ***
   That experiment overwrites a predicted concept with its GROUND-TRUTH
   value and asks whether accuracy improves. It requires attribute GT on
   the images being classified, and the measured intersection is empty,
   so it is not performable in this repo at all.

   What is performable: overwrite a concept with a CHOSEN value and ask
   how the prediction moves. That answers a strictly weaker question -
   "how much does f() lean on this concept" - not "were the concepts
   right". It cannot report an accuracy gain, because there is no
   ground-truth target to move towards. Any accuracy computed against a
   synthetic concept value would be meaningless and none is computed.
   ==================================================================="""
def sensitivity_sweep(C, f, attribute, suffix='present', values=(0.0, 1.0)):
    """Set one concept dimension to each value in `values` for every test image,
    re-run f, and count how the predicted-class distribution moves.

    Note the signature: this function is NOT GIVEN THE LABELS. That is deliberate
    and structural, not an oversight - with a synthetic concept value there is no
    correct answer to score against, so no accuracy can be computed here even by
    mistake. Compare Koh et al.'s intervention, which is defined by moving the
    concept towards its TRUE value and therefore does have an accuracy to report."""
    k = ATTRIBUTES.index(attribute) * len(CONCEPT_SUFFIXES) + CONCEPT_SUFFIXES.index(suffix)
    base = f.predict(C)
    rows = []
    for v in values:
        Cx = C.copy(); Cx[:, k] = v
        pred = f.predict(Cx)
        rows.append({
            'concept set to': f'{CONCEPT_NAMES[k]} = {v}',
            'changed vs baseline': int((pred != base).sum()),
            'fraction changed': float((pred != base).mean()),
            **{f'-> {DIAGNOSES[c]}': int((pred == c).sum()) for c in range(N_CLASS)},
        })
    return pd.DataFrame(rows), int(k)

def leverage_table(C, f):
    """How much does each of the 20 dimensions matter to f()?
    Two complementary readings, neither of which needs any ground truth:
      * flip rate  - set the dimension to its 10th and 90th percentile across the
                     test set and count how often the predicted class changes.
      * |coef|     - the mean absolute standardised coefficient over the 7 classes.
    A dimension with high leverage AND low fidelity is the dangerous combination."""
    base = f.predict(C)
    clf = f.named_steps['clf']
    coefs = clf.coef_
    if coefs.shape[0] == 1:
        coefs = np.vstack([-coefs[0], coefs[0]])
    rows = []
    for k, name in enumerate(CONCEPT_NAMES):
        lo, hi = np.percentile(C[:, k], [10, 90])
        flips = 0
        for v in (lo, hi):
            Cx = C.copy(); Cx[:, k] = v
            flips += int((f.predict(Cx) != base).sum())
        rows.append({'concept': name,
                     'flip rate (p10/p90 sweep)': flips / (2.0 * len(C)),
                     'mean |coef|': float(np.abs(coefs[:, k]).mean())})
    return pd.DataFrame(rows).sort_values('flip rate (p10/p90 sweep)', ascending=False)

C_te = C_pred[idx_te]

print('--- per-attribute presence sweep (synthetic, not a correction) ---')
for a in ATTRIBUTES:
    sw, _ = sensitivity_sweep(C_te, f_sequential, a, 'present')
    print()
    print(ATTR_LABELS[a])
    print(sw.to_string(index=False))

print()
print('--- leverage of all', N_CONCEPT, 'dimensions ---')
lev_df = leverage_table(C_te, f_sequential)
print(lev_df.to_string(index=False))
lev_df.to_csv(os.path.join(save_dir, 'concept_leverage.csv'), index=False)

# **Risk triage for the 7-class pipeline**

Clean failure attribution needs ground-truth concepts on the classified images, which Route B has
not got. What it can have is the two halves of the question, measured on different image sets and
joined by an assumption:

* **fidelity** - on held-out Task 1-2 images - how often is `g` right about this attribute?
* **leverage** - on the Task 3 test split - how much does `f` lean on it?

Their product is a risk score: an attribute that `g` gets wrong *and* `f` relies on heavily is where
a confident, fluent, wrong rationale comes from. That is a genuinely useful thing to have, and it is
not error attribution - the two columns come from different institutions' images and the join
between them is the unverified transfer assumption again.


In [ ]:
"""====================================================================
   FAILURE ATTRIBUTION, as far as an empty intersection allows.

   The clean version - "was this error a concept error or a reasoning
   error?" - needs GT concepts on the classified images and is not
   available here. What IS available is the two halves of the question,
   measured on different image sets, joined by the transfer assumption:

     fidelity  (task-2 held-out split) : how often is g() right about
                                         this attribute?
     leverage  (task-3 test split)     : how much does f() lean on it?

   The product is the risk: a concept g() gets wrong AND f() relies on
   heavily is where a confident, fluent, wrong rationale comes from.
   This is a TRIAGE tool, not a measurement of error attribution, and
   the two columns come from different images.
   ==================================================================="""
if fid_mask_df is not None:
    """collapse the 4 dimensions of each attribute to its max leverage"""
    lev_by_attr = {}
    for a in ATTRIBUTES:
        prefix = f'{a}__'
        sub = lev_df[lev_df['concept'].str.startswith(prefix)]
        lev_by_attr[ATTR_LABELS[a]] = float(sub['flip rate (p10/p90 sweep)'].max())

    risk = fid_mask_df[['Attribute', 'mask Jaccard (pooled)', 'presence AUC']].copy()
    risk['max leverage on f()'] = risk['Attribute'].map(lev_by_attr)
    """unreliability = 1 - presence AUC scaled to [0,1] from the 0.5 chance floor"""
    risk['unreliability (1 - AUC skill)'] = 1.0 - ((risk['presence AUC'] - 0.5) / 0.5).clip(0, 1)
    risk['RISK = unreliability x leverage'] = (risk['unreliability (1 - AUC skill)']
                                               * risk['max leverage on f()'])
    risk = risk.sort_values('RISK = unreliability x leverage', ascending=False)
    print(risk.to_string(index=False))
    risk.to_csv(os.path.join(save_dir, 'concept_risk.csv'), index=False)
    print()
    print('How to read the top row: that attribute is the one whose rationale')
    print('sentences deserve the least trust. It is also the first thing to fix -')
    print('either by improving g() on that channel or by removing the attribute')
    print('from the bottleneck and accepting a narrower explanation.')
    print()
    print('Reminder: the fidelity columns are from ISIC_00xxxxx images and the')
    print('leverage column from ISIC_002-003xxxx images. Joining them assumes g()')
    print('transfers between the two batches. Nothing here measures that.')
else:
    print('no fidelity table - the risk triage needs both halves.')

# **Honest comparison**

Side by side against the black box, with the trade named in both directions - and with every row
that is *not* comparable to the 0.615 baseline marked as such in its own column, because a table
like this is exactly where a 3-class number on memorised images would otherwise quietly become a
headline.


In [ ]:
"""====================================================================
   SIDE BY SIDE. Only the black-box row carries a number, because it is
   the only one that has been measured. Every other cell is NaN until
   this notebook is run end to end.
   ==================================================================="""
BLACKBOX_ACC = 0.615   # measured: task3-lesion-classification.ipynb, small Sequential
                       # CNN on 32x32 input, 7 classes, resampled to 500/class.

rows = [{
    'model': 'black box: small CNN, 32x32 pixels (task 3)',
    'dataset': 'task 3 / HAM10000',
    'classes': 7,
    'bottleneck': 'none - 3072 raw pixel values',
    'test accuracy': BLACKBOX_ACC,
    'explanation': 'saliency map at best; nothing in clinical vocabulary',
    'per-concept audit': False,
    'concept intervention': False,
    'comparable to the 0.615 row': '(is the row)',
}]

for key in ['sequential', 'sequential_tree']:
    if key in results:
        r = results[key]
        rows.append({
            'model': r['name'], 'dataset': 'task 3 / HAM10000', 'classes': N_CLASS,
            'bottleneck': f'{N_CONCEPT} named concepts',
            'test accuracy': r['accuracy'],
            'explanation': 'signed per-concept contributions / decision path',
            'per-concept audit': 'only by transfer from task-2 images',
            'concept intervention': 'synthetic only',
            'comparable to the 0.615 row': 'same classes, DIFFERENT split',
        })

## bench rows are 3-class, on images g() trained on. Kept in the table because
## hiding them would be worse, and flagged so they cannot be quietly misread.
for key in ['independent', 'independent_oracle']:
    if key in results:
        r = results[key]
        rows.append({
            'model': r['name'], 'dataset': 'AUDIT BENCH (task 1-2)',
            'classes': len(DIAG_A) if AUDIT_OK else None,
            'bottleneck': f'{N_CONCEPT} named concepts',
            'test accuracy': r['accuracy'],
            'explanation': 'signed per-concept contributions',
            'per-concept audit': 'yes, on these exact images',
            'concept intervention': 'yes, real (GT concepts)',
            'comparable to the 0.615 row': 'NO - 3 classes, g() trained on these',
        })

comparison = pd.DataFrame(rows)
print(comparison.to_string(index=False))
comparison.to_csv(os.path.join(save_dir, 'comparison.csv'), index=False)

print()
print('THREE CAVEATS, all of which have to survive into any write-up.')
print()
print('1. The 0.615 row is NOT a like-for-like baseline. It was trained on a')
print('   class-resampled 32x32 set with its own split; the CBM rows use this')
print("   notebook's split over whatever the join produced. The comparison is")
print('   indicative of the interpretability tradeoff, not a controlled experiment.')
print('   Making it controlled means retraining that CNN on exactly these ids.')
print()
print('2. Beating 0.615 would NOT show that 5 dermoscopic attributes suffice to')
print('   diagnose 7 classes. It might only show that a 32x32 CNN is weak - which')
print('   the repo README already concedes (see "What I would do differently", #1).')
print('   The honest comparator is a properly trained image model, which does not')
print('   exist in this repo.')
print()
print('3. Four of the seven classes (BCC, AKIEC, BKL, DF, VASC) are NOT melanocytic')
print('   lesions, and the five ISIC attributes are melanocytic pattern criteria.')
print('   Whatever accuracy the CBM reaches on those classes is being achieved')
print('   through concepts that do not describe them - which is either leakage or')
print('   luck, and in both cases the rationale for those classes is not to be')
print('   trusted. Read the per-class report, not the headline accuracy.')
print()
print('4. Any AUDIT BENCH row above is 3-class, on legacy-archive images that g() was')
print('   TRAINED on. It exists to show what the architecture can do when the data')
print('   cooperates - real intervention, real fidelity, real failure attribution -')
print('   and it is not an accuracy claim about this pipeline. Do not quote it as one.')

"""floors, so that "accuracy" has a scale at all"""
maj = np.bincount(y_all[idx_te], minlength=N_CLASS).max() / len(idx_te)
print()
print('majority-class accuracy on this test split :', maj)
print('uniform random over 7 classes              :', 1.0 / N_CLASS)
print('melanocytic share of the test split        :',
      float(np.isin([DIAGNOSES[c] for c in y_all[idx_te]], list(MELANOCYTIC)).mean()))

# **Limitations**

Ordered roughly by how much they should change your reading of anything above.

**1. Nothing here is clinically validated, and none of it is a medical device.** This is a
personal learning project. No clinician has reviewed the concept definitions, the geometric
features, the `CLINICAL_NOTE` wording, or a single rationale. The clinical sentences are a
non-clinician's paraphrase of published literature, with sources, and they describe textbook
associations for named structures - not findings about a patient. Nothing in this notebook should
inform a decision about a person.

**2. The concepts are unqualified, and the qualifier is the diagnosis.** The single deepest
problem. Three of the five ISIC attributes map onto a 7-point checklist criterion only in their
*atypical* / *irregular* form, and ISIC's masks record no such distinction; a typical pigment
network is a benign sign while an atypical one carries OR 2.8 for melanoma. The `asymmetry` and
`n_blobs` features are an *attempt* to recover the qualifier from geometry and **that attempt is
unvalidated** - it is the first thing a dermatologist should be asked to assess. The remaining two
attributes, negative network and milia-like cysts, are not 7-point criteria at all.

**3. Five attributes are an incomplete description of dermoscopic diagnosis, provably so.** Four
of the seven target classes (BCC, AKIEC, BKL, DF, VASC) are not melanocytic lesions, and these
five are melanocytic pattern criteria - the bottleneck barely describes them. Meanwhile the
strongest single predictors in Carrera et al.'s reliability study (marked architectural disorder
OR 6.6, pattern asymmetry OR 4.9) are absent from the five, and blue-white veil and atypical
vascular pattern - two of the three *major* 7-point criteria - are absent too. Napoles et al.
prove a **92.1% accuracy ceiling for any hard concept bottleneck** built on derm7pt's seven
*qualified* criteria, with 54% of melanomas falling in inconsistent boundary regions
(<https://arxiv.org/html/2604.19323>). That exact number does not transfer to ISIC's five
unqualified masks, but the direction does, and it is worse here, not better.

**4. Expect an accuracy drop against a real black box.** The comparison table's 0.615 row is a
32x32 CNN and a weak comparator. Against a properly trained image model the bottleneck should
cost accuracy: Koh et al. saw CBMs lose on CUB (0.199 vs 0.175 error) whenever the concept set
was incomplete, and PCBM on skin data lost 8.5 AUROC points on their ISIC task (0.821 -> 0.736)
until a residual side channel bought it back (0.801) at the cost of the interpretability
guarantee. An incomplete concept set is exactly the condition here.

**5. Concept leakage.** Even trained sequentially with a frozen `g`, the continuous dimensions
(`area_frac` especially) can carry information about the image far beyond the named attribute -
Mahinpei et al. get 69% on a task whose concepts are provably irrelevant. So a large coefficient
is **not** evidence that the concept relates to the diagnosis; it is evidence that this `g`'s
channel is useful to this `f`. The `present` dimensions are the more trustworthy half of the
vector for exactly this reason, and the risk-triage table exists to flag where the two diverge.

**6. Concept fidelity is measured on the wrong images.** Consequence of the empty intersection:
`g` is audited on Task 1-2 images and used on Task 3 images. Those are not just different splits -
they are different institutions (legacy MSK/UDA/SONIC vs Vienna), different licenses, and
different acquisition resolutions (206 distinct sizes vs a uniform 600x450). The transfer is
assumed and nowhere measured, and it is the load-bearing assumption in the notebook. Two further
reasons the fidelity figures may be optimistic: the reconstructed Task-2 held-out split is not
guaranteed identical to the one task 2 actually trained on, and a mirrored ground-truth directory
may bundle the train/val/test splits together (2594 + 100 + 1000 = 3694), in which case the
"held-out" images were partly seen by `g`.

**7. No true concept intervention on the 7-class task.** The clearest demonstration of why this
architecture is worth building - correct a concept to its ground-truth value and watch accuracy
improve - is not performable on Route B, because no Task 3 image has attribute ground truth. It is
performable on the 3-class audit bench and is run there. But the bench has its own problem: `g` was
trained on most of those images, so its intervention and fidelity numbers show the concept
extractor on its own training distribution and are an upper bound, not an estimate. For the 7-class
pipeline what remains is a synthetic sensitivity analysis, which cannot report an accuracy gain and
does not attempt to.

**8. HAM10000's population, and who these concepts were validated on.** HAM10000 is 10,015
dermatoscopic images "collected from different populations acquired and stored by different
modalities" (<https://arxiv.org/abs/1803.10417>), and it ships **no Fitzpatrick or skin-tone
labels**, so the distribution cannot even be checked from the release. Analyses that estimate
skin tone from the pixels via individual typology angle find that the great majority of images in
ISIC-family benchmarks fall in the lighter-skin range (Kinyanjui et al., MICCAI 2020 -
<https://krvarshney.github.io/pubs/KinyanjuiOCCPSV_miccai2020.pdf>). Two consequences: whatever
`g` learns about how these five structures *look* is calibrated on predominantly light skin, and
the dermoscopic criteria themselves - the odds ratios quoted throughout - come from cohorts with
the same skew. An interpretable rationale is not a fair one. Naming the concepts makes the
reasoning auditable; it does nothing to make the concepts valid for people who were not in the
data. Related: MONET's authors audited >70,000 ISIC images and found concept-label correlations
that differ between benign and malignant classes
(<https://www.nature.com/articles/s41591-024-02887-x>) - i.e. concept-level confounding in this
dataset family.

**9. Single split, no error bars.** One stratified 75/25 split, seed 42. Every number this
notebook produces is a point estimate with no variance, which is the same criticism the repo
README already makes of Task 3's 0.615. A stratified k-fold over the concept matrix would be
cheap here - the matrix is 20 columns wide and the expensive part (`g` inference) is already
cached - and it is not done.

**10. `g` is a weak concept extractor, by the standards of the task.** The world's best Task 2
model averages 0.292 Jaccard. Everything above inherits that. Improving the pipeline starts with
`g`, not with `f`: a better label predictor on unreliable concepts produces more confident
fiction, not better diagnoses.

**What I would actually do next**, in order: (1) get a dermatologist to review the
`CLINICAL_NOTE` block and the geometric-qualifier claim; (2) rebuild this on **derm7pt**, which
annotates the same criteria *with* their typical/atypical qualifiers
(<https://derm.cs.sfu.ca>) - the qualifier is the thing ISIC is missing and the thing the
diagnosis depends on; (3) use **MONET** to generate concept scores without needing masks at all,
which also removes the empty-intersection problem entirely, since MONET scores any image; (4)
only then worry about accuracy.
